# AML Gremlin Showcase on HugeGraph

 This notebook implements anti-money-laundering (AML) analysis as executable Gremlin traversals in HugeGraph, using the following schema:

- vertex label: `account`
- edge label: `transfer`
- account properties: `bank`, `acct`
- transfer properties: `ts`, `amount_paid`, `amount_recv`, `pay_currency`,
  `recv_currency`, `pay_format`, `is_laundering`, `row`

The focus here is executing and observing Gremlin query behavior in HugeGraph.
To stay stable on large datasets, queries are intentionally scoped with `limit(...)`
and seed vertices.


## Step 0 - Configuration

- Start the cloud-storage stack first.
- Put `LI-Large_Trans.csv` and `HI-Large_accounts.csv` under `docker/cloud-storage/notebooks/data`
  (or another candidate path shown in config); this notebook can load the data in Step 2.


## Step 1 - Helpers


In [1]:
# Step 1 can run standalone even if Step 0/config cells were not executed yet.
import json
import time
import requests
import html
from IPython.display import display, HTML

HG_HOST = "127.0.0.1"  # HugeGraph server host.
HG_PORT = 8080  # HugeGraph server port.
GRAPH = "hugegraph"  # Graph name used by this notebook.

BASE = f"http://{HG_HOST}:{HG_PORT}/graphs/{GRAPH}"  # Graph REST base endpoint.
GREMLIN = f"http://{HG_HOST}:{HG_PORT}/gremlin"  # Gremlin HTTP endpoint.

GREMLIN_EVAL_TIMEOUT_MS = 180_000  # Default Gremlin evaluation timeout per query.
ROW_PRINT_LIMIT = 20  # Max rows printed by show().
ALIAS_STABILITY_TIMEOUT_SEC = 180  # Max seconds to wait for a stable alias binding.
ALIAS_STABILITY_POLL_SEC = 2  # Seconds between alias-binding probes.
ALIAS_STABILITY_CONSECUTIVE_SUCCESSES = 3  # Successes required before pinning alias.
ALIAS_STABILITY_QUERY = "g.V().limit(1).count()"  # Probe query used for alias checks.
GREMLIN_PINNED_ALIAS = None  # Optional fixed alias (None enables dynamic probing/fallback).

session = requests.Session()
session.auth = ("admin", "admin")
session.headers.update({"Content-Type": "application/json"})

GREMLIN_ALIAS_CANDIDATES = [
    f"__g_DEFAULT-{GRAPH}",
    f"__g_{GRAPH}",
    GRAPH,
]


def ordered_alias_candidates():
    if GREMLIN_PINNED_ALIAS in GREMLIN_ALIAS_CANDIDATES:
        return [
            GREMLIN_PINNED_ALIAS,
            *[a for a in GREMLIN_ALIAS_CANDIDATES if a != GREMLIN_PINNED_ALIAS],
        ]
    return list(GREMLIN_ALIAS_CANDIDATES)

TRANSIENT_GREMLIN_ERRORS = (
    "refCnt: 0, decrement: 1",
    "SocketTimeoutException",
    "Read timed out",
    "ProcessingException",
    "RejectedExecutionException",
    "connect timed out",
    "connection reset",
    "connection refused",
)

ALIAS_FALLBACK_ERRORS = (
    "No such property: g",
    "Could not rebind [g]",
    "TraversalSource global bindings",
)


def _contains_error_token(message, tokens):
    text = str(message or "").lower()
    return any(token.lower() in text for token in tokens)


def is_transient_gremlin_error(message):
    return _contains_error_token(message, TRANSIENT_GREMLIN_ERRORS)


def needs_alias_fallback(message):
    return _contains_error_token(message, ALIAS_FALLBACK_ERRORS)


def is_timeout_gremlin_error(message):
    return _contains_error_token(
        message,
        (
            "timeoutexception",
            "evaluation exceeded",
            "evaluationtimeout",
        ),
    )


def _check(r):
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:800]}")
    return r


def _post_gremlin(body, retries=2, timeout=240):
    last_err = ""
    for attempt in range(retries + 1):
        try:
            r = session.post(GREMLIN, data=json.dumps(body), timeout=timeout)
        except requests.RequestException as e:
            err = str(e)
            last_err = err
            if is_transient_gremlin_error(err) and attempt < retries:
                time.sleep(1 + attempt)
                continue
            return None, err

        if r.status_code < 400:
            return r.json()["result"]["data"], ""

        err = r.text[:800]
        last_err = err
        if is_transient_gremlin_error(err) and attempt < retries:
            time.sleep(1 + attempt)
            continue
        return None, err
    return None, last_err


def gremlin(query, bindings=None, eval_timeout_ms=GREMLIN_EVAL_TIMEOUT_MS):
    body = {"gremlin": query, "language": "gremlin-groovy"}
    if bindings:
        body["bindings"] = bindings
    if eval_timeout_ms is not None:
        body["evaluationTimeout"] = int(eval_timeout_ms)

    if GREMLIN_PINNED_ALIAS:
        body_with_alias = dict(body)
        body_with_alias["aliases"] = {"g": GREMLIN_PINNED_ALIAS}
        result, err = _post_gremlin(body_with_alias)
    else:
        result, err = _post_gremlin(body)
    if result is not None:
        return result

    # If a pinned alias executed but failed for a non-alias reason (for example timeout),
    # surface that root cause directly instead of masking it with alias-rebind noise.
    if GREMLIN_PINNED_ALIAS and not needs_alias_fallback(err):
        raise RuntimeError(
            f"Gremlin query failed with pinned alias '{GREMLIN_PINNED_ALIAS}': {err}"
        )

    if not needs_alias_fallback(err) and not is_transient_gremlin_error(err):
        raise RuntimeError(f"Gremlin query failed: {err}")

    last_err = err
    last_timeout_err = err if is_timeout_gremlin_error(err) else ""
    alias_rounds = 2
    for round_idx in range(alias_rounds):
        for alias in ordered_alias_candidates():
            body_with_alias = dict(body)
            body_with_alias["aliases"] = {"g": alias}
            result, alias_err = _post_gremlin(body_with_alias)
            if result is not None:
                return result

            last_err = alias_err
            if is_timeout_gremlin_error(alias_err):
                last_timeout_err = alias_err
            if needs_alias_fallback(alias_err) or is_transient_gremlin_error(alias_err):
                continue
            raise RuntimeError(f"Gremlin query failed with alias '{alias}': {alias_err}")

        if round_idx + 1 < alias_rounds and is_transient_gremlin_error(last_err):
            time.sleep(1 + round_idx)

    if last_timeout_err:
        raise RuntimeError(
            "Gremlin query timed out after alias fallback. Tried aliases "
            f"{ordered_alias_candidates()}. Last timeout: {last_timeout_err}"
        )

    raise RuntimeError(
        "Gremlin query failed after alias fallback. Tried aliases "
        f"{ordered_alias_candidates()}. Last error: {last_err}"
    )


def server_up():
    try:
        return session.get(f"http://{HG_HOST}:{HG_PORT}/graphs", timeout=5).status_code == 200
    except Exception:
        return False


def _probe_alias(alias, query=ALIAS_STABILITY_QUERY, eval_timeout_ms=10_000):
    body = {
        "gremlin": query,
        "language": "gremlin-groovy",
        "aliases": {"g": alias},
        "evaluationTimeout": int(eval_timeout_ms),
    }
    result, err = _post_gremlin(body, retries=1, timeout=20)
    return result is not None, err


def wait_for_gremlin_alias_binding(
    timeout_sec=ALIAS_STABILITY_TIMEOUT_SEC,
    poll_sec=ALIAS_STABILITY_POLL_SEC,
    required_consecutive=ALIAS_STABILITY_CONSECUTIVE_SUCCESSES,
):
    global GREMLIN_PINNED_ALIAS

    deadline = time.time() + int(timeout_sec)
    stable_alias = None
    consecutive = 0
    checks = 0
    last_errors = {}

    while time.time() < deadline:
        checks += 1
        winner = None

        for alias in ordered_alias_candidates():
            ok, err = _probe_alias(alias)
            if ok:
                winner = alias
                break
            last_errors[alias] = err

        if winner:
            if winner == stable_alias:
                consecutive += 1
            else:
                stable_alias = winner
                consecutive = 1

            print(
                f"Alias probe {checks}: using '{winner}' "
                f"({consecutive}/{int(required_consecutive)})"
            )

            if consecutive >= int(required_consecutive):
                GREMLIN_PINNED_ALIAS = winner
                GREMLIN_ALIAS_CANDIDATES[:] = [
                    winner,
                    *[a for a in GREMLIN_ALIAS_CANDIDATES if a != winner],
                ]
                print(f"Gremlin alias binding is stable. Pinned alias: {winner}")
                return winner
        else:
            stable_alias = None
            consecutive = 0
            if checks % 5 == 0:
                preview = "; ".join(
                    f"{a}: {str(e)[:120]}" for a, e in list(last_errors.items())[:3]
                )
                print(
                    "Alias probe pending: no alias succeeded yet; "
                    f"retrying in {int(poll_sec)}s. Last errors: {preview}"
                )

        time.sleep(int(poll_sec))

    raise TimeoutError(
        "Timed out waiting for stable Gremlin alias binding. "
        f"Tried aliases {ordered_alias_candidates()} for {int(timeout_sec)}s"
    )


def _format_row(row, max_len=260):
    try:
        text = json.dumps(row, ensure_ascii=True)
    except Exception:
        text = str(row)
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


_WRAP_STYLE = (
    "white-space:pre-wrap !important; word-break:break-all !important; "
    "overflow-wrap:anywhere !important; text-align:left !important; margin:0;"
)


def _render_result_html(rows, row_limit, error):
    if error:
        return f"<pre style='{_WRAP_STYLE}'>FAILED: {_esc(error)}</pre>"
    if not isinstance(rows, list):
        return f"<pre style='{_WRAP_STYLE}'>{_esc(str(rows))}</pre>"
    shown = rows[: int(row_limit)]
    if not shown:
        return "<i>(no rows)</i>"
    lines = [_esc(_format_row(r)) for r in shown]
    body = "<br>".join(lines)
    if len(rows) > int(row_limit):
        body += f"<br><i>... ({len(rows) - int(row_limit)} more rows)</i>"
    return f"<div style='font-family:monospace; {_WRAP_STYLE}'>{body}</div>"


def _esc(text):
    return html.escape(str(text))


def show(query, title="", description="", bindings=None,
         eval_timeout_ms=GREMLIN_EVAL_TIMEOUT_MS, row_limit=ROW_PRINT_LIMIT):
    # Renders a 3-row table for one Gremlin call: description (what/why),
    # the Gremlin query itself, and the formatted result (or error) --
    # so a reader can see intent, query, and outcome together in one place.
    # Jupyter's notebook CSS (.rendered_html td) right-aligns <td> text and
    # collapses whitespace by default, so every cell below sets its own
    # text-align/white-space explicitly rather than relying on the default.
    try:
        rows = gremlin(query, bindings=bindings, eval_timeout_ms=eval_timeout_ms)
        error = None
    except Exception as e:
        rows = []
        error = str(e)

    if error:
        status = "FAILED"
    elif isinstance(rows, list):
        status = f"{len(rows)} row(s)"
    else:
        status = "1 value"

    desc_html = _esc(title)
    if description:
        desc_html += "<br>" + _esc(description).replace("\n", "<br>")
    result_html = _render_result_html(rows, row_limit, error)

    # table-layout:fixed + a <colgroup> pins the label column's width so the
    # browser can't just widen the value column to fit an unbroken token
    # (e.g. a long Gremlin query with no spaces) -- it's forced to wrap
    # instead. !important beats notebook themes that right-align/no-wrap
    # <td> by default.
    label_style = (
        "text-align:left !important; vertical-align:top; white-space:nowrap; "
        "background:rgba(127,127,127,0.15); border:1px solid rgba(127,127,127,0.35); "
        "padding:4px 8px;"
    )
    value_style = (
        "text-align:left !important; vertical-align:top; "
        "border:1px solid rgba(127,127,127,0.35); padding:4px 8px; "
        "white-space:pre-wrap !important; word-break:break-all !important; "
        "overflow-wrap:anywhere !important;"
    )
    query_style = value_style + " font-family:monospace;"

    display(HTML(f"""
    <table style="border-collapse:collapse; table-layout:fixed; width:100%;
                  margin:6px 0 14px 0; font-size:13px; text-align:left;">
      <colgroup><col style="width:130px"><col></colgroup>
      <tr>
        <th style="{label_style}">Description</th>
        <td style="{value_style}">{desc_html}</td>
      </tr>
      <tr>
        <th style="{label_style}">Gremlin Query</th>
        <td style="{query_style}">{_esc(query)}</td>
      </tr>
      <tr>
        <th style="{label_style}">Result ({status})</th>
        <td style="{value_style}">{result_html}</td>
      </tr>
    </table>
    """))

## Step 1.5 - Preflight alias-binding gate

Blocks execution until one Gremlin traversal-source alias is repeatedly successful.
This reduces intermittent `No such property: g` failures later in the notebook.


In [2]:
wait_for_gremlin_alias_binding()


Alias probe 1: using '__g_DEFAULT-hugegraph' (1/3)
Alias probe 2: using '__g_DEFAULT-hugegraph' (2/3)
Alias probe 3: using '__g_DEFAULT-hugegraph' (3/3)
Gremlin alias binding is stable. Pinned alias: __g_DEFAULT-hugegraph


'__g_DEFAULT-hugegraph'

In [3]:
import json
import time
import csv
import shutil
import subprocess
from pathlib import Path
import requests

HG_HOST = "127.0.0.1"  # HugeGraph server host.
HG_PORT = 8080  # HugeGraph server port.
GRAPH = "hugegraph"  # Graph name used by this notebook.

BASE = f"http://{HG_HOST}:{HG_PORT}/graphs/{GRAPH}"  # Graph REST base endpoint.
GREMLIN = f"http://{HG_HOST}:{HG_PORT}/gremlin"  # Gremlin HTTP endpoint.

GREMLIN_EVAL_TIMEOUT_MS = 180_000  # Default Gremlin evaluation timeout per query.
ROW_PRINT_LIMIT = 20  # Max rows printed by show().
ALIAS_STABILITY_TIMEOUT_SEC = 180  # Max seconds to wait for a stable alias binding.
ALIAS_STABILITY_POLL_SEC = 2  # Seconds between alias-binding probes.
ALIAS_STABILITY_CONSECUTIVE_SUCCESSES = 3  # Successes required before pinning alias.
ALIAS_STABILITY_QUERY = "g.V().limit(1).count()"  # Probe query used for alias checks.
# Set this to a specific alias string to force pinning (for example '__g_DEFAULT-hugegraph').
# Keep None to allow dynamic alias fallback/probing.
GREMLIN_PINNED_ALIAS = None

# Local AML CSV inputs (auto-detect common locations).
TRANS_CSV_NAME = "LI-Large_Trans.csv"  # Primary AML transactions CSV file name.
ACCOUNTS_CSV_NAME = "HI-Large_accounts.csv"  # Optional accounts CSV file name.
DATA_DIR_CANDIDATES = [
    Path("./data"),
    Path("./docker/cloud-storage/notebooks/data"),
    Path("./notebooks/data"),
    Path("../notebooks/data"),
    Path("."),
]
# Pick the first candidate directory that already contains the transactions CSV.
DATA_DIR = next(
    (d for d in DATA_DIR_CANDIDATES if (d / TRANS_CSV_NAME).exists()),
    DATA_DIR_CANDIDATES[0],
)
TRANS_CSV = DATA_DIR / TRANS_CSV_NAME  # Resolved transactions CSV path.
ACCOUNTS_CSV = DATA_DIR / ACCOUNTS_CSV_NAME  # Resolved accounts CSV path.

# Loader controls.
LOAD_ENABLED = True  # Master switch for Step 2 schema/data load.
LOAD_IF_GRAPH_EMPTY = True  # Load only if graph has no transfer data.
FORCE_RELOAD = True  # Force load even when data already exists.
CLEAR_GRAPH_BEFORE_LOAD = False  # Clear graph data before loading.
LOAD_ROWS = 5000000  # Max transaction rows to load; set None to load full file.
VERTEX_BATCH = 2000  # Vertex REST batch size for insert calls.
EDGE_BATCH = 2000  # Edge REST batch size for insert calls.
LOAD_PROGRESS_EVERY_ROWS = 100_000  # Print loader progress every N rows.
MAX_SEEN_ACCOUNTS = 4_000_000  # Soft cap for in-memory account dedup set.
SCHEMA_HTTP_TIMEOUT_SEC = 20  # Timeout for schema REST requests.
SCHEMA_TASK_TIMEOUT_SEC = 120  # Max wait time for async schema tasks.
SCHEMA_TASK_POLL_SEC = 2  # Poll interval for schema task status checks.
KAGGLE_AUTO_DOWNLOAD = True  # Download AML CSVs if missing.
KAGGLE_DATASET = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"  # Kaggle dataset slug.
KAGGLE_TOKEN_PATH = Path.home() / ".kaggle" / "kaggle.json"  # Kaggle API token path.

# Scoped traversal knobs (edit values here to override defaults for this notebook run).

# Deep-traversal safety controls (used by Step 15d scoped 6-hop query).
HOP6_OUT_LIMIT = 1200  # Max outgoing edges expanded per hop in 6-hop scoped traversal.
HOP6_RESULT_LIMIT = 50000  # Cap on unique 6-hop results after dedup.

# Scoped query safety controls (tune for speed vs coverage).
ROOT_COUNT_SCOPE_LIMIT = 1000  # Scope for root sample counts (Step 1a/1b).
HASNOT_SCOPE_LIMIT = 10000  # Scope for hasNot() check (Step 2d).
GROUPCOUNT_ACCOUNT_SCOPE_LIMIT = 300  # Vertex scope before currency groupCount (Step 5e).
GROUPCOUNT_EDGE_SCOPE_LIMIT = 5000  # Edge scope after account expansion (Step 5e).
RANGE_SCOPE_LIMIT = 2000  # Scope for ordering/range paging examples (Step 7a/7b/7c).
DEDUP_SCOPE_LIMIT = 200  # Scope for neighbor dedup count (Step 7d).
RANKING_SCOPE_LIMIT = 3000  # Scope for flagged account ranking (Step 14a).
HOP8_OUT_LIMIT = 1000  # Max outgoing edges expanded per hop in 8-hop traversal (Step 15b).
HOP8_RESULT_LIMIT = 50000  # Cap on unique 8-hop results after dedup (Step 15b).
ALIAS_PAIR_SCOPE_LIMIT = 200  # Scope for as/select pair sampling (Step 8a).
FLAGGED_PROJECT_SCOPE_LIMIT = 1000  # Scope for flagged_out projection ranking (Step 8b).
COMPOUND_WHERE_SCOPE_LIMIT = 1000  # Scope for compound where() examples (Step 9).
BANK_GROUP_SCOPE_LIMIT = 5000  # Scope for bank-level account groupCount (Step 10a).

print("Graph REST:", BASE)
print("Gremlin endpoint:", GREMLIN)
print("Data directory:", DATA_DIR)
print("Transactions CSV:", TRANS_CSV)
print("Accounts CSV:", ACCOUNTS_CSV)
print("Load enabled:", LOAD_ENABLED)
print("Load rows:", "full file" if LOAD_ROWS is None else int(LOAD_ROWS))
print("Kaggle auto download:", KAGGLE_AUTO_DOWNLOAD)
print("Kaggle dataset:", KAGGLE_DATASET)
print("Step 15d HOP6_OUT_LIMIT:", int(HOP6_OUT_LIMIT))
print("Step 15d HOP6_RESULT_LIMIT:", int(HOP6_RESULT_LIMIT))
print(
    "Scoped safety limits:",
    {
        "ROOT_COUNT_SCOPE_LIMIT": int(ROOT_COUNT_SCOPE_LIMIT),
        "HASNOT_SCOPE_LIMIT": int(HASNOT_SCOPE_LIMIT),
        "GROUPCOUNT_ACCOUNT_SCOPE_LIMIT": int(GROUPCOUNT_ACCOUNT_SCOPE_LIMIT),
        "GROUPCOUNT_EDGE_SCOPE_LIMIT": int(GROUPCOUNT_EDGE_SCOPE_LIMIT),
        "RANGE_SCOPE_LIMIT": int(RANGE_SCOPE_LIMIT),
        "DEDUP_SCOPE_LIMIT": int(DEDUP_SCOPE_LIMIT),
        "RANKING_SCOPE_LIMIT": int(RANKING_SCOPE_LIMIT),
        "HOP8_OUT_LIMIT": int(HOP8_OUT_LIMIT),
        "HOP8_RESULT_LIMIT": int(HOP8_RESULT_LIMIT),
        "ALIAS_PAIR_SCOPE_LIMIT": int(ALIAS_PAIR_SCOPE_LIMIT),
        "FLAGGED_PROJECT_SCOPE_LIMIT": int(FLAGGED_PROJECT_SCOPE_LIMIT),
        "COMPOUND_WHERE_SCOPE_LIMIT": int(COMPOUND_WHERE_SCOPE_LIMIT),
        "BANK_GROUP_SCOPE_LIMIT": int(BANK_GROUP_SCOPE_LIMIT),
    },
)


Graph REST: http://127.0.0.1:8080/graphs/hugegraph
Gremlin endpoint: http://127.0.0.1:8080/gremlin
Data directory: data
Transactions CSV: data/LI-Large_Trans.csv
Accounts CSV: data/HI-Large_accounts.csv
Load enabled: True
Load rows: 5000000
Kaggle auto download: True
Kaggle dataset: ealtman2019/ibm-transactions-for-anti-money-laundering-aml
Step 15d HOP6_OUT_LIMIT: 1200
Step 15d HOP6_RESULT_LIMIT: 50000
Scoped safety limits: {'ROOT_COUNT_SCOPE_LIMIT': 1000, 'HASNOT_SCOPE_LIMIT': 10000, 'GROUPCOUNT_ACCOUNT_SCOPE_LIMIT': 300, 'GROUPCOUNT_EDGE_SCOPE_LIMIT': 5000, 'RANGE_SCOPE_LIMIT': 2000, 'DEDUP_SCOPE_LIMIT': 200, 'RANKING_SCOPE_LIMIT': 3000, 'HOP8_OUT_LIMIT': 1000, 'HOP8_RESULT_LIMIT': 50000, 'ALIAS_PAIR_SCOPE_LIMIT': 200, 'FLAGGED_PROJECT_SCOPE_LIMIT': 1000, 'COMPOUND_WHERE_SCOPE_LIMIT': 1000, 'BANK_GROUP_SCOPE_LIMIT': 5000}


## Step 1.6 - CSV preflight (required for end-to-end runs)

This step always checks whether `LI-Large_Trans.csv` and
`HI-Large_accounts.csv` already exist.

- If files exist, it only reports status (no re-download).
- If files are missing and `KAGGLE_AUTO_DOWNLOAD=True`, it downloads from Kaggle.
- If files are missing and `KAGGLE_AUTO_DOWNLOAD=False`, Step 2 fails fast with a clear error.

Requirements:
- `kaggle` CLI installed in the kernel environment
- API token at `~/.kaggle/kaggle.json`


In [4]:
def _file_size_mb(path):
    return f"{path.stat().st_size / 1e6:.1f} MB"


def show_aml_csv_status():
    for csv_path in (TRANS_CSV, ACCOUNTS_CSV):
        status = "found" if csv_path.exists() else "MISSING"
        size = _file_size_mb(csv_path) if csv_path.exists() else "-"
        print(f"  [{status}] {csv_path} ({size})")


def download_aml_csvs_from_kaggle(dataset=KAGGLE_DATASET, data_dir=DATA_DIR, files=None):
    files = list(files or [TRANS_CSV_NAME, ACCOUNTS_CSV_NAME])
    kaggle_bin = shutil.which("kaggle")
    if kaggle_bin is None:
        raise RuntimeError(
            "kaggle CLI not found. Install with: pip install kaggle"
        )
    if not KAGGLE_TOKEN_PATH.exists():
        raise RuntimeError(
            f"Kaggle token not found at {KAGGLE_TOKEN_PATH}. "
            "Create it from your Kaggle account API page."
        )

    data_dir.mkdir(parents=True, exist_ok=True)

    for fname in files:
        target = data_dir / fname
        if target.exists():
            print(f"  [found] {target} ({_file_size_mb(target)})")
            continue

        cmd = [
            kaggle_bin, "datasets", "download", dataset,
            "-f", fname,
            "-p", str(data_dir),
            "--unzip",
        ]
        print("Running:", " ".join(cmd))
        try:
            # Keep Kaggle stdout/stderr attached so notebook shows live download progress
            subprocess.run(cmd, check=True, text=True)
        except subprocess.CalledProcessError as e:
            err = (e.stderr or e.stdout or str(e)).strip()
            raise RuntimeError(
                f"Kaggle download failed for {fname}: {err[-500:]}"
            ) from e

        if not target.exists():
            raise RuntimeError(
                f"Kaggle command completed but expected file is missing: {target}"
            )
        print(f"  [downloaded] {target} ({_file_size_mb(target)})")


def ensure_aml_csvs(download_if_missing=KAGGLE_AUTO_DOWNLOAD):
    missing = [p for p in (TRANS_CSV, ACCOUNTS_CSV) if not p.exists()]
    if missing and download_if_missing:
        print("Missing AML CSV files -> attempting Kaggle download...")
        download_aml_csvs_from_kaggle()
    show_aml_csv_status()


ensure_aml_csvs(download_if_missing=KAGGLE_AUTO_DOWNLOAD)


  [found] data/LI-Large_Trans.csv (16742.5 MB)
  [found] data/HI-Large_accounts.csv (147.7 MB)


## Step 2 - Schema and idempotent data load

Creates the AML schema if needed and loads `LI-Large_Trans.csv`
into HugeGraph only when required by loader settings.

With defaults, if transfer data already exists in the graph,
the load is skipped. Set `FORCE_RELOAD=True` to load again.


In [5]:
def _schema_get(url, action):
    try:
        return session.get(url, timeout=SCHEMA_HTTP_TIMEOUT_SEC)
    except requests.RequestException as e:
        raise RuntimeError(
            f"[schema] {action} failed "
            f"(GET timeout={int(SCHEMA_HTTP_TIMEOUT_SEC)}s): {e}"
        ) from e


def _schema_post(url, body, action):
    try:
        return session.post(
            url,
            data=json.dumps(body),
            timeout=SCHEMA_HTTP_TIMEOUT_SEC,
        )
    except requests.RequestException as e:
        raise RuntimeError(
            f"[schema] {action} failed "
            f"(POST timeout={int(SCHEMA_HTTP_TIMEOUT_SEC)}s): {e}"
        ) from e


def create_property_key(name, data_type="TEXT", cardinality="SINGLE"):
    print(f"[schema] pk {name}")
    if _schema_get(f"{BASE}/schema/propertykeys/{name}", f"check pk {name}").status_code == 200:
        print(f"[schema] pk {name}: exists")
        return
    _check(_schema_post(
        f"{BASE}/schema/propertykeys",
        {"name": name, "data_type": data_type, "cardinality": cardinality},
        f"create pk {name}",
    ))
    print(f"[schema] pk {name}: created")


def create_vertex_label(name, props, id_strategy="AUTOMATIC", primary_keys=None):
    print(f"[schema] vl {name}")
    if _schema_get(f"{BASE}/schema/vertexlabels/{name}", f"check vl {name}").status_code == 200:
        print(f"[schema] vl {name}: exists")
        return
    body = {
        "name": name,
        "id_strategy": id_strategy,
        "properties": props,
        "nullable_keys": [p for p in props if p not in (primary_keys or [])],
    }
    if primary_keys:
        body["primary_keys"] = primary_keys
    _check(_schema_post(f"{BASE}/schema/vertexlabels", body, f"create vl {name}"))
    print(f"[schema] vl {name}: created")


def create_edge_label(name, source, target, props, frequency="SINGLE", sort_keys=None):
    print(f"[schema] el {name}")
    if _schema_get(f"{BASE}/schema/edgelabels/{name}", f"check el {name}").status_code == 200:
        print(f"[schema] el {name}: exists")
        return
    body = {
        "name": name,
        "source_label": source,
        "target_label": target,
        "properties": props,
        "frequency": frequency,
        "nullable_keys": [p for p in props if p not in (sort_keys or [])],
    }
    if sort_keys:
        body["sort_keys"] = sort_keys
    _check(_schema_post(f"{BASE}/schema/edgelabels", body, f"create el {name}"))
    print(f"[schema] el {name}: created")


def wait_task(task_id, timeout_sec=SCHEMA_TASK_TIMEOUT_SEC):
    timeout_sec = int(timeout_sec)
    deadline = time.time() + timeout_sec
    last_status = "UNKNOWN"
    last_reported_status = None
    unknown_count = 0
    print(f"[schema] index task id: {task_id} (timeout={timeout_sec}s)")
    while time.time() < deadline:
        payload = _check(
            _schema_get(f"{BASE}/tasks/{task_id}", f"check task {task_id}")
        ).json()
        # HugeGraph returns task fields like task_status/task_progress in task APIs.
        task = payload.get("task") if isinstance(payload, dict) else None
        if not isinstance(task, dict):
            task = payload if isinstance(payload, dict) else {}

        raw_status = (
            task.get("task_status") or
            task.get("status") or
            payload.get("task_status") if isinstance(payload, dict) else ""
        )
        progress = task.get("task_progress") or task.get("progress") or ""
        last_status = str(raw_status or "").upper() or "UNKNOWN"

        if last_status == "UNKNOWN":
            unknown_count += 1
            if unknown_count >= 3:
                keys = sorted(task.keys()) if isinstance(task, dict) else []
                raise RuntimeError(
                    f"[schema] task {task_id} response missing status fields after {unknown_count} polls. "
                    f"Task keys={keys}, payload={payload}"
                )
        else:
            unknown_count = 0

        if last_status != last_reported_status:
            suffix = f" progress={progress}" if progress != "" else ""
            print(f"[schema] task {task_id}: {last_status}{suffix}")
            last_reported_status = last_status

        if last_status in {"SUCCESS", "FAILED", "CANCELED", "CANCELLED"}:
            if last_status != "SUCCESS":
                raise RuntimeError(f"[schema] task {task_id} failed with status={last_status}: {task}")
            return
        time.sleep(int(SCHEMA_TASK_POLL_SEC))
    raise TimeoutError(
        f"[schema] task {task_id} timed out after {timeout_sec}s "
        f"(last status={last_status})"
    )


def create_index_label(name, base_type, base_value, index_type, fields):
    print(f"[schema] il {name}")
    if _schema_get(f"{BASE}/schema/indexlabels/{name}", f"check il {name}").status_code == 200:
        print(f"[schema] il {name}: exists")
        return
    body = {
        "name": name,
        "base_type": base_type,
        "base_value": base_value,
        "index_type": index_type,
        "fields": fields,
    }
    resp = _check(
        _schema_post(f"{BASE}/schema/indexlabels", body, f"create il {name}")
    ).json()
    task_id = resp.get("task_id")
    if task_id not in (None, 0, "0"):
        wait_task(task_id)
    print(f"[schema] il {name}: created")


def ensure_aml_schema():
    create_property_key("bank", "TEXT")
    create_property_key("acct", "TEXT")
    create_property_key("ts", "TEXT")
    create_property_key("amount_paid", "DOUBLE")
    create_property_key("amount_recv", "DOUBLE")
    create_property_key("pay_currency", "TEXT")
    create_property_key("recv_currency", "TEXT")
    create_property_key("pay_format", "TEXT")
    create_property_key("is_laundering", "INT")
    create_property_key("row", "LONG")

    create_vertex_label("account", ["bank", "acct"], id_strategy="CUSTOMIZE_STRING")
    create_edge_label(
        "transfer", "account", "account",
        [
            "ts", "amount_paid", "amount_recv", "pay_currency", "recv_currency",
            "pay_format", "is_laundering", "row",
        ],
        frequency="MULTIPLE", sort_keys=["row"],
    )
    create_index_label(
        "transfer_by_is_laundering",
        "EDGE_LABEL",
        "transfer",
        "SECONDARY",
        ["is_laundering"],
    )
    create_index_label(
        "account_by_bank",
        "VERTEX_LABEL",
        "account",
        "SECONDARY",
        ["bank"],
    )


def is_retriable_write_error(message):
    text = str(message or "").lower()
    return (
        is_transient_gremlin_error(text) or
        "cluster is not ready" in text or
        "active stores" in text or
        "partition_fault_type_unknown" in text
    )


def post_batch_with_retry(url, payload, label, attempts=12, base_sleep=2, max_sleep=20):
    for attempt in range(1, int(attempts) + 1):
        try:
            resp = session.post(url, data=json.dumps(payload), timeout=180)
        except requests.RequestException as e:
            if attempt >= attempts:
                raise
            sleep_sec = min(int(max_sleep), int(base_sleep) * attempt)
            print(f"  retry {label} attempt {attempt}/{attempts} after {sleep_sec}s ({e})")
            time.sleep(sleep_sec)
            continue

        if resp.status_code < 400:
            return

        err = f"HTTP {resp.status_code}: {resp.text[:800]}"
        retriable = resp.status_code >= 500 or is_retriable_write_error(resp.text)
        if retriable and attempt < attempts:
            sleep_sec = min(int(max_sleep), int(base_sleep) * attempt)
            print(f"  retry {label} attempt {attempt}/{attempts} after {sleep_sec}s ({err})")
            time.sleep(sleep_sec)
            continue
        _check(resp)


def vid(bank, acct):
    return f"{bank}:{acct}"


def normalize_headers(raw_headers):
    seen = {}
    normalized = []
    for h in (raw_headers or []):
        name = str(h or "").strip()
        idx = seen.get(name, 0)
        normalized.append(name if idx == 0 else f"{name}.{idx}")
        seen[name] = idx + 1
    return normalized


def row_to_dict(headers, row):
    vals = [str(v) for v in row]
    if len(vals) < len(headers):
        vals.extend([""] * (len(headers) - len(vals)))
    return {h: vals[i] for i, h in enumerate(headers)}


def to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0


def to_int(x):
    s = str(x).strip()
    try:
        return int(float(s)) if s else 0
    except (TypeError, ValueError):
        return 0


def resolve_columns(headers):
    cols = {c.lower().strip(): c for c in headers}

    def col(*cands):
        for c in cands:
            if c.lower() in cols:
                return cols[c.lower()]
        raise KeyError(f"None of {cands} found in {headers}")

    return {
        "ts": col("Timestamp"),
        "fbank": col("From Bank"),
        "facct": col("Account"),
        "tbank": col("To Bank"),
        "tacct": col("Account.1", "Account 1"),
        "arecv": col("Amount Received"),
        "rcur": col("Receiving Currency"),
        "apaid": col("Amount Paid"),
        "pcur": col("Payment Currency"),
        "pfmt": col("Payment Format"),
        "flag": col("Is Laundering"),
    }


def make_edge(row, row_id, c):
    from_bank = str(row.get(c["fbank"], ""))
    from_acct = str(row.get(c["facct"], ""))
    to_bank = str(row.get(c["tbank"], ""))
    to_acct = str(row.get(c["tacct"], ""))
    return {
        "label": "transfer",
        "outV": vid(from_bank, from_acct), "outVLabel": "account",
        "inV": vid(to_bank, to_acct), "inVLabel": "account",
        "properties": {
            "ts": str(row.get(c["ts"], "")),
            "amount_paid": to_float(row.get(c["apaid"], "")),
            "amount_recv": to_float(row.get(c["arecv"], "")),
            "pay_currency": str(row.get(c["pcur"], "")),
            "recv_currency": str(row.get(c["rcur"], "")),
            "pay_format": str(row.get(c["pfmt"], "")),
            "is_laundering": to_int(row.get(c["flag"], "")),
            "row": int(row_id),
        },
    }


def graph_has_transfer_data():
    try:
        result = gremlin("g.E().hasLabel('transfer').limit(1).count()", eval_timeout_ms=20_000)
        return bool(result and int(result[0]) > 0)
    except Exception:
        return False


def clear_graph_data():
    print("Clearing graph data...")
    resp = session.delete(
        f"{BASE}/clear",
        params={"confirm_message": "I'm sure to delete all data"},
        timeout=120,
    )
    _check(resp)


def load_aml_transactions(max_rows=LOAD_ROWS):
    if not TRANS_CSV.exists():
        raise FileNotFoundError(f"Transactions CSV not found: {TRANS_CSV}")

    seen_accounts = set()
    vertex_batch = []
    edge_batch = []
    rows_loaded = 0
    vertices_loaded = 0
    edges_loaded = 0
    start_ts = time.time()

    with TRANS_CSV.open("r", newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        headers = normalize_headers(next(reader, []))
        c = resolve_columns(headers)

        for row_id, raw_row in enumerate(reader):
            if max_rows is not None and rows_loaded >= int(max_rows):
                break

            row = row_to_dict(headers, raw_row)
            from_id = vid(str(row.get(c["fbank"], "")), str(row.get(c["facct"], "")))
            to_id = vid(str(row.get(c["tbank"], "")), str(row.get(c["tacct"], "")))

            if from_id not in seen_accounts:
                seen_accounts.add(from_id)
                vertex_batch.append({
                    "label": "account",
                    "id": from_id,
                    "properties": {
                        "bank": str(row.get(c["fbank"], "")),
                        "acct": str(row.get(c["facct"], "")),
                    },
                })

            if to_id not in seen_accounts:
                seen_accounts.add(to_id)
                vertex_batch.append({
                    "label": "account",
                    "id": to_id,
                    "properties": {
                        "bank": str(row.get(c["tbank"], "")),
                        "acct": str(row.get(c["tacct"], "")),
                    },
                })

            edge_batch.append(make_edge(row, row_id, c))

            if len(vertex_batch) >= int(VERTEX_BATCH):
                post_batch_with_retry(f"{BASE}/graph/vertices/batch", vertex_batch, "vertices")
                vertices_loaded += len(vertex_batch)
                vertex_batch.clear()

            if len(edge_batch) >= int(EDGE_BATCH):
                if vertex_batch:
                    post_batch_with_retry(f"{BASE}/graph/vertices/batch", vertex_batch, "vertices")
                    vertices_loaded += len(vertex_batch)
                    vertex_batch.clear()
                post_batch_with_retry(
                    f"{BASE}/graph/edges/batch?check_vertex=false",
                    edge_batch,
                    "edges",
                )
                edges_loaded += len(edge_batch)
                edge_batch.clear()

            rows_loaded += 1
            if rows_loaded % int(LOAD_PROGRESS_EVERY_ROWS) == 0:
                elapsed = max(time.time() - start_ts, 1e-9)
                print(
                    f"  rows={rows_loaded:,} inserted(v={vertices_loaded:,},e={edges_loaded:,}) "
                    f"avg_rows/s={rows_loaded / elapsed:,.1f}"
                )

            if MAX_SEEN_ACCOUNTS is not None and len(seen_accounts) >= int(MAX_SEEN_ACCOUNTS):
                seen_accounts.clear()

    if vertex_batch:
        post_batch_with_retry(f"{BASE}/graph/vertices/batch", vertex_batch, "vertices")
        vertices_loaded += len(vertex_batch)
    if edge_batch:
        post_batch_with_retry(f"{BASE}/graph/edges/batch?check_vertex=false", edge_batch, "edges")
        edges_loaded += len(edge_batch)

    elapsed = max(time.time() - start_ts, 1e-9)
    print(
        f"Loaded rows={rows_loaded:,} vertices={vertices_loaded:,} edges={edges_loaded:,} "
        f"in {elapsed:.1f}s"
    )


def maybe_load_data():
    if not LOAD_ENABLED:
        print("LOAD_ENABLED=False -> skipping schema/data load")
        return

    print("Ensuring AML schema...")
    try:
        ensure_aml_schema()
    except Exception as e:
        raise RuntimeError(
            "Schema setup failed fast. Check HugeGraph schema/task endpoints and logs. "
            f"Cause: {e}"
        ) from e
    print("AML schema is ready")

    if CLEAR_GRAPH_BEFORE_LOAD:
        clear_graph_data()

    has_data = graph_has_transfer_data()
    print("Existing transfer data:", has_data)

    should_load = FORCE_RELOAD or (LOAD_IF_GRAPH_EMPTY and not has_data)
    if not should_load:
        print("Skipping load (graph already has data; set FORCE_RELOAD=True to load again)")
        return

    if "ensure_aml_csvs" not in globals():
        raise RuntimeError(
            "CSV preflight helpers are not defined. "
            "Run Step 1.6 or execute the notebook from top to bottom."
        )

    ensure_aml_csvs(download_if_missing=KAGGLE_AUTO_DOWNLOAD)

    if not TRANS_CSV.exists():
        raise FileNotFoundError(
            f"Missing transactions CSV: {TRANS_CSV}. Place {TRANS_CSV_NAME} under {DATA_DIR}."
        )

    if ACCOUNTS_CSV.exists():
        print(f"Found accounts CSV (optional): {ACCOUNTS_CSV}")
    else:
        print(f"Accounts CSV not found (optional): {ACCOUNTS_CSV}")

    load_aml_transactions(max_rows=LOAD_ROWS)


maybe_load_data()


Ensuring AML schema...
[schema] pk bank
[schema] pk bank: created
[schema] pk acct
[schema] pk acct: created
[schema] pk ts
[schema] pk ts: created
[schema] pk amount_paid
[schema] pk amount_paid: created
[schema] pk amount_recv
[schema] pk amount_recv: created
[schema] pk pay_currency
[schema] pk pay_currency: created
[schema] pk recv_currency
[schema] pk recv_currency: created
[schema] pk pay_format
[schema] pk pay_format: created
[schema] pk is_laundering
[schema] pk is_laundering: created
[schema] pk row
[schema] pk row: created
[schema] vl account
[schema] vl account: created
[schema] el transfer
[schema] el transfer: created
[schema] il transfer_by_is_laundering
[schema] index task id: 1 (timeout=120s)
[schema] task 1: RUNNING
[schema] task 1: SUCCESS
[schema] il transfer_by_is_laundering: created
[schema] il account_by_bank
[schema] index task id: 2 (timeout=120s)
[schema] task 2: RUNNING
[schema] task 2: SUCCESS
[schema] il account_by_bank: created
AML schema is ready
Existing 

## Step 3 - Health check and dynamic seeds


In [6]:
assert server_up(), "HugeGraph server is not reachable on :8080"
print("HugeGraph is up. Graphs:", session.get(f"http://{HG_HOST}:{HG_PORT}/graphs").json())
print("Pinned Gremlin alias:", GREMLIN_PINNED_ALIAS)
print("Gremlin sanity check 1+1:", gremlin("1+1"))

sample_accounts = show(
    "g.V().hasLabel('account').limit(3).project('id','bank','acct').by(id).by('bank').by('acct')",
    "Sample account vertices",
    description="Health-check preview: first 3 'account' vertices in the graph, "
                 "confirming the load actually produced readable account data before "
                 "any of the numbered example queries run.",
)
sample_transfers = show(
    "g.E().hasLabel('transfer').limit(3).project('id','from','to','amount','flag')"
    ".by(id).by(outV().id()).by(inV().id()).by('amount_paid').by('is_laundering')",
    "Sample transfer edges",
    description="Health-check preview: first 3 'transfer' edges, confirming edge "
                 "data (with amount/flag properties) loaded correctly alongside the "
                 "accounts above.",
)

SEED_OUT = (gremlin("g.V().hasLabel('account').where(outE('transfer')).id().limit(1)") or [None])[0]
SEED_IN = (gremlin("g.V().hasLabel('account').where(inE('transfer')).id().limit(1)") or [None])[0]
SEED_ANY = (gremlin("g.V().hasLabel('account').id().limit(1)") or [None])[0]
FLAGGED_SEED = (
    gremlin("g.E().hasLabel('transfer').has('is_laundering',1).outV().id().limit(1)") or [None]
)[0]

SEED = SEED_OUT or SEED_IN or SEED_ANY
print("SEED_OUT:", SEED_OUT)
print("SEED_IN:", SEED_IN)
print("SEED_ANY:", SEED_ANY)
print("FLAGGED_SEED:", FLAGGED_SEED)
print("Active seed:", SEED)

SEED_BANK = None
if SEED is not None:
    _seed_vertex = gremlin(
        "g.V(s).project('id','bank','acct').by(id).by('bank').by('acct')",
        bindings={"s": SEED}
    )
    if _seed_vertex:
        SEED_BANK = _seed_vertex[0].get("bank")
        print("Seed vertex:", _seed_vertex[0])


HugeGraph is up. Graphs: {'graphs': ['hugegraph']}
Pinned Gremlin alias: None
Gremlin sanity check 1+1: [2]


Description,"Sample account verticesHealth-check preview: first 3 'account' vertices in the graph, confirming the load actually produced readable account data before any of the numbered example queries run."
Gremlin Query,"g.V().hasLabel('account').limit(3).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (3 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:800047930"", ""bank"": ""000"", ""acct"": ""800047930""}{""id"": ""000:80006C140"", ""bank"": ""000"", ""acct"": ""80006C140""}"


Description,"Sample transfer edgesHealth-check preview: first 3 'transfer' edges, confirming edge data (with amount/flag properties) loaded correctly alongside the accounts above."
Gremlin Query,"g.E().hasLabel('transfer').limit(3).project('id','from','to','amount','flag').by(id).by(outV().id()).by(inV().id()).by('amount_paid').by('is_laundering')"
Result (3 row(s)),"{""id"": ""S000:8000474C0>2>2>27H>S000:8000474C0"", ""from"": ""000:8000474C0"", ""to"": ""000:8000474C0"", ""amount"": 11.21, ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46fR4>S000:8000474C0"", ""from"": ""000:8000474C0"", ""to"": ""000:8000474C0"", ""amount"": 65.15, ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46fR5>S000:8000474C0"", ""from"": ""000:8000474C0"", ""to"": ""000:8000474C0"", ""amount"": 579.25, ""flag"": 0}"


SEED_OUT: 000:8000474C0
SEED_IN: 000:8000474C0
SEED_ANY: 000:8000474C0
FLAGGED_SEED: 000:800D81CB0
Active seed: 000:8000474C0
Seed vertex: {'id': '000:8000474C0', 'bank': '000', 'acct': '8000474C0'}


## 1 - Root steps (`g.V()`, `g.E()`, `g.V(id)`)


In [7]:
show(
    "g.V().count()",
    "1x Total vertex count",
    description="Total vertex count across the whole graph (every label, not just "
                 "'account') -- a coarse sanity check that data actually loaded.",
)
show(
    "g.E().count()",
    "1y Total edge count",
    description="Total edge count across the whole graph (every label, not just "
                 "'transfer') -- paired with the vertex count this gives a rough "
                 "density sense (avg edges per vertex) for the loaded dataset.",
)
show(
    f"g.V().hasLabel('account').limit({int(ROOT_COUNT_SCOPE_LIMIT)}).count()",
    "1a Root: scoped account count",
    description="Count of 'account' vertices only, with a limit() applied before the "
                 "count so a very large graph doesn't force an unscoped full-label scan.",
)
show(
    f"g.E().hasLabel('transfer').limit({int(ROOT_COUNT_SCOPE_LIMIT)}).count()",
    "1b Root: scoped transfer count",
    description="Count of 'transfer' edges only, same scoped-before-count pattern as 1a.",
)
if SEED is not None:
    show(
        "g.V(s).project('id','bank','acct').by(id).by('bank').by('acct')",
        "1c Root: g.V(id)",
        description=f"g.V(id): direct lookup of the account resolved by SEED (id={SEED}) "
                     f"by its vertex id -- the cheapest possible read since it skips any "
                     f"label/property scan and goes straight to the vertex. Projects "
                     f"id/bank/acct for a quick look.",
        bindings={"s": SEED},
    )


Description,"1x Total vertex countTotal vertex count across the whole graph (every label, not just 'account') -- a coarse sanity check that data actually loaded."
Gremlin Query,g.V().count()
Result (1 row(s)),1717587


Description,"1y Total edge countTotal edge count across the whole graph (every label, not just 'transfer') -- paired with the vertex count this gives a rough density sense (avg edges per vertex) for the loaded dataset."
Gremlin Query,g.E().count()
Result (1 row(s)),5000001


Description,"1a Root: scoped account countCount of 'account' vertices only, with a limit() applied before the count so a very large graph doesn't force an unscoped full-label scan."
Gremlin Query,g.V().hasLabel('account').limit(1000).count()
Result (1 row(s)),1000


Description,"1b Root: scoped transfer countCount of 'transfer' edges only, same scoped-before-count pattern as 1a."
Gremlin Query,g.E().hasLabel('transfer').limit(1000).count()
Result (1 row(s)),1000


Description,1c Root: g.V(id)g.V(id): direct lookup of the account resolved by SEED (id=000:8000474C0) by its vertex id -- the cheapest possible read since it skips any label/property scan and goes straight to the vertex. Projects id/bank/acct for a quick look.
Gremlin Query,"g.V(s).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (1 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}"


## 2 - Filter steps (`has`, `hasLabel`, `hasId`, `hasNot`, `is`)


In [8]:
if SEED_BANK is not None:
    show(
        "g.V().hasLabel('account').has('bank', b).limit(10)"
        ".project('id','bank','acct').by(id).by('bank').by('acct')",
        "2a has + hasLabel",
        description=f"has('bank', b) + hasLabel('account'): every account issued by the "
                     f"bank resolved by SEED_BANK ({SEED_BANK}) -- filtering the vertex set "
                     f"down to a single issuing institution before inspecting individual "
                     f"accounts.",
        bindings={"b": SEED_BANK},
    )

show(
    "g.E().hasLabel('transfer').has('is_laundering',1).limit(10)"
    ".project('from','to','amount','currency').by(outV().id()).by(inV().id())"
    ".by('amount_paid').by('pay_currency')",
    "2b has on edge property",
    description="has('is_laundering', 1) on an edge property: transfers explicitly "
                 "flagged as laundering activity, listing sender, receiver, amount, and "
                 "currency for each -- this is the core 'show me the suspicious "
                 "transactions' query.",
)

if SEED is not None:
    show(
        "g.V().hasId(s).project('id','bank','acct').by(id).by('bank').by('acct')",
        "2c hasId",
        description=f"hasId(s): same lookup as 1c for the account resolved by SEED "
                     f"(id={SEED}), but expressed as a filter predicate over g.V() instead "
                     f"of the direct g.V(id) addressing form -- useful when the id needs to "
                     f"be combined with other has()/hasLabel() filters.",
        bindings={"s": SEED},
    )

show(
    f"g.V().hasLabel('account').limit({int(HASNOT_SCOPE_LIMIT)}).hasNot('bank').count()",
    "2d hasNot on scoped subset",
    description="hasNot('bank'): a data-quality check -- accounts within a scoped "
                 "sample that are missing the 'bank' property entirely (incomplete/bad "
                 "records).",
)

if SEED is not None:
    show(
        "g.V(s).outE('transfer').values('amount_paid').is(gt(0)).limit(10)",
        "2e is(gt(...))",
        description=f"is(gt(0)): outgoing transfer amounts strictly greater than zero for "
                     f"the account resolved by SEED (id={SEED}), i.e. excluding any "
                     f"zero/negative-amount transfers that would indicate bad or reversed "
                     f"data rather than real money movement.",
        bindings={"s": SEED},
    )


Description,"2a has + hasLabelhas('bank', b) + hasLabel('account'): every account issued by the bank resolved by SEED_BANK (000) -- filtering the vertex set down to a single issuing institution before inspecting individual accounts."
Gremlin Query,"g.V().hasLabel('account').has('bank', b).limit(10).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (10 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:800047930"", ""bank"": ""000"", ""acct"": ""800047930""}{""id"": ""000:80006C140"", ""bank"": ""000"", ""acct"": ""80006C140""}{""id"": ""000:80006DFE0"", ""bank"": ""000"", ""acct"": ""80006DFE0""}{""id"": ""000:80006DFE1"", ""bank"": ""000"", ""acct"": ""80006DFE1""}{""id"": ""000:80006F150"", ""bank"": ""000"", ""acct"": ""80006F150""}{""id"": ""000:80006FE20"", ""bank"": ""000"", ""acct"": ""80006FE20""}{""id"": ""000:8000719B0"", ""bank"": ""000"", ""acct"": ""8000719B0""}{""id"": ""000:800071D00"", ""bank"": ""000"", ""acct"": ""800071D00""}{""id"": ""000:800071D50"", ""bank"": ""000"", ""acct"": ""800071D50""}"


Description,"2b has on edge propertyhas('is_laundering', 1) on an edge property: transfers explicitly flagged as laundering activity, listing sender, receiver, amount, and currency for each -- this is the core 'show me the suspicious transactions' query."
Gremlin Query,"g.E().hasLabel('transfer').has('is_laundering',1).limit(10).project('from','to','amount','currency').by(outV().id()).by(inV().id()).by('amount_paid').by('pay_currency')"
Result (10 row(s)),"{""from"": ""000:800D81CB0"", ""to"": ""002258:800D81DF0"", ""amount"": 2681.26, ""currency"": ""US Dollar""}{""from"": ""000:8028D5120"", ""to"": ""0114703:808454400"", ""amount"": 493.15, ""currency"": ""Euro""}{""from"": ""001:818DB9EF0"", ""to"": ""0028488:818DBA4B0"", ""amount"": 243.47, ""currency"": ""Euro""}{""from"": ""002:803644610"", ""to"": ""012:8036446B0"", ""amount"": 10317.61, ""currency"": ""Yuan""}{""from"": ""003:8114BCC20"", ""to"": ""0044261:81126EEB0"", ""amount"": 930730228.37, ""currency"": ""Yen""}{""from"": ""004:81C4E0F70"", ""to"": ""004:81B4513F0"", ""amount"": 895840.49, ""currency"": ""Ruble""}{""from"": ""005:80008D800"", ""to"": ""0022177:80824F9D0"", ""amount"": 660159.3, ""currency"": ""Yen""}{""from"": ""005:80824CF90"", ""to"": ""013:80008C270"", ""amount"": 257267.97, ""currency"": ""Yen""}{""from"": ""005:80C1F0980"", ""to"": ""023:811AF7CE0"", ""amount"": 362288.79, ""currency"": ""Yen""}{""from"": ""006:812E98450"", ""to"": ""025:812E986D0"", ""amount"": 692423.6, ""currency"": ""Rupee""}"


Description,"2c hasIdhasId(s): same lookup as 1c for the account resolved by SEED (id=000:8000474C0), but expressed as a filter predicate over g.V() instead of the direct g.V(id) addressing form -- useful when the id needs to be combined with other has()/hasLabel() filters."
Gremlin Query,"g.V().hasId(s).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (1 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}"


Description,2d hasNot on scoped subsethasNot('bank'): a data-quality check -- accounts within a scoped sample that are missing the 'bank' property entirely (incomplete/bad records).
Gremlin Query,g.V().hasLabel('account').limit(10000).hasNot('bank').count()
Result (1 row(s)),0


Description,"2e is(gt(...))is(gt(0)): outgoing transfer amounts strictly greater than zero for the account resolved by SEED (id=000:8000474C0), i.e. excluding any zero/negative-amount transfers that would indicate bad or reversed data rather than real money movement."
Gremlin Query,g.V(s).outE('transfer').values('amount_paid').is(gt(0)).limit(10)
Result (10 row(s)),11.2165.15579.25289.99912.44120864.2921145.5215718.124216.0229855.52


## 3 - Hop steps (`out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV`)


In [9]:
if SEED is not None:
    show(
        "g.V(s).out('transfer').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')",
        "3a out('transfer')",
        description=f"out('transfer'): follow the outgoing transfer edges of the account "
                     f"resolved by SEED (id={SEED}) to the accounts it paid -- i.e. 'where "
                     f"did SEED's money go', one hop.",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).in('transfer').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')",
        "3b in('transfer')",
        description=f"in('transfer'): follow edges pointing into the account resolved by "
                     f"SEED (id={SEED}) -- i.e. 'who paid SEED', the mirror image of 3a.",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).both('transfer').dedup().limit(10).project('id','bank','acct')"
        ".by(id).by('bank').by('acct')",
        "3c both('transfer')",
        description=f"both('transfer'): every counterparty connected to the account "
                     f"resolved by SEED (id={SEED}) regardless of whether they paid SEED or "
                     f"were paid by it, deduplicated so an account that both sent and "
                     f"received doesn't show up twice.",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).outE('transfer').limit(10).project('id','amount','currency','flag')"
        ".by(id).by('amount_paid').by('pay_currency').by('is_laundering')",
        "3d outE('transfer')",
        description=f"outE('transfer'): the outgoing transfer edges of the account "
                     f"resolved by SEED (id={SEED}) themselves (not the neighbor vertices) "
                     f"-- exposes per-transaction detail (amount, currency, laundering "
                     f"flag) for every payment SEED made.",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).inE('transfer').limit(10).project('id','amount','currency','flag')"
        ".by(id).by('amount_paid').by('pay_currency').by('is_laundering')",
        "3e inE('transfer')",
        description=f"inE('transfer'): the incoming transfer edges of the account "
                     f"resolved by SEED (id={SEED}), same per-transaction detail as 3d but "
                     f"for payments received.",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).bothE('transfer').limit(10).project('id','amount').by(id).by('amount_paid')",
        "3f bothE('transfer')",
        description=f"bothE('transfer'): every transfer edge touching the account "
                     f"resolved by SEED (id={SEED}) in either direction, combining 3d and "
                     f"3e into one edge set.",
        bindings={"s": SEED},
    )

show(
    "g.E().hasLabel('transfer').limit(10).outV().project('id','bank','acct').by(id).by('bank').by('acct')",
    "3g outV() from edge root",
    description="outV(): starting from a batch of transfer edges (not from one seed), "
                 "jump to each edge's source vertex -- the payer for every sampled "
                 "transfer.",
)

show(
    "g.E().hasLabel('transfer').limit(10).inV().project('id','bank','acct').by(id).by('bank').by('acct')",
    "3h inV() from edge root",
    description="inV(): same edge-root starting point as 3g, but jump to each edge's "
                 "destination vertex -- the payee for every sampled transfer.",
)

show(
    "g.E().hasLabel('transfer').limit(10).bothV().dedup().project('id','bank','acct')"
    ".by(id).by('bank').by('acct')",
    "3i bothV() from edge root",
    description="bothV(): from the same sampled edges, jump to both endpoints and "
                 "deduplicate -- every account that appears anywhere in the sample, "
                 "whether as payer or payee.",
)

if SEED is not None:
    show(
        "g.V(s).bothE('transfer').limit(10).otherV().project('id','bank','acct')"
        ".by(id).by('bank').by('acct')",
        "3j otherV()",
        description=f"otherV(): starting from the own edges of the account resolved by "
                     f"SEED (id={SEED}) (not an arbitrary edge batch), hop to whichever "
                     f"endpoint is NOT SEED -- a direction-agnostic way to list SEED's "
                     f"counterparties without needing to know in advance whether each "
                     f"edge is incoming or outgoing.",
        bindings={"s": SEED},
    )


Description,"3a out('transfer')out('transfer'): follow the outgoing transfer edges of the account resolved by SEED (id=000:8000474C0) to the accounts it paid -- i.e. 'where did SEED's money go', one hop."
Gremlin Query,"g.V(s).out('transfer').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (10 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""0154487:8212A2430"", ""bank"": ""0154487"", ""acct"": ""8212A2430""}"


Description,"3b in('transfer')in('transfer'): follow edges pointing into the account resolved by SEED (id=000:8000474C0) -- i.e. 'who paid SEED', the mirror image of 3a."
Gremlin Query,"g.V(s).in('transfer').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (4 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}"


Description,"3c both('transfer')both('transfer'): every counterparty connected to the account resolved by SEED (id=000:8000474C0) regardless of whether they paid SEED or were paid by it, deduplicated so an account that both sent and received doesn't show up twice."
Gremlin Query,"g.V(s).both('transfer').dedup().limit(10).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (8 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""0154487:8212A2430"", ""bank"": ""0154487"", ""acct"": ""8212A2430""}{""id"": ""01208:80013C440"", ""bank"": ""01208"", ""acct"": ""80013C440""}{""id"": ""00214:800122740"", ""bank"": ""00214"", ""acct"": ""800122740""}{""id"": ""019662:82B58B510"", ""bank"": ""019662"", ""acct"": ""82B58B510""}{""id"": ""020:80022BC40"", ""bank"": ""020"", ""acct"": ""80022BC40""}"


Description,"3d outE('transfer')outE('transfer'): the outgoing transfer edges of the account resolved by SEED (id=000:8000474C0) themselves (not the neighbor vertices) -- exposes per-transaction detail (amount, currency, laundering flag) for every payment SEED made."
Gremlin Query,"g.V(s).outE('transfer').limit(10).project('id','amount','currency','flag').by(id).by('amount_paid').by('pay_currency').by('is_laundering')"
Result (10 row(s)),"{""id"": ""S000:8000474C0>2>2>27H>S000:8000474C0"", ""amount"": 11.21, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46fR4>S000:8000474C0"", ""amount"": 65.15, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46fR5>S000:8000474C0"", ""amount"": 579.25, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46wl2>S0122705:80DB95010"", ""amount"": 289.99, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46wl3>S0122705:80DB95010"", ""amount"": 912.44, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>48gLn>S00867:80124BD60"", ""amount"": 120864.29, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>48gLo>S00867:80124BD60"", ""amount"": 21145.52, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>48gLp>S00867:80124BD60"", ""amount"": 15718.12, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>48gLq>S00867:80124BD60"", ""amount"": 4216.02, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>4ATjX>S0154487:8212A2430"", ""amount"": 29855.52, ""currency"": ""US Dollar"", ""flag"": 0}"


Description,"3e inE('transfer')inE('transfer'): the incoming transfer edges of the account resolved by SEED (id=000:8000474C0), same per-transaction detail as 3d but for payments received."
Gremlin Query,"g.V(s).inE('transfer').limit(10).project('id','amount','currency','flag').by(id).by('amount_paid').by('pay_currency').by('is_laundering')"
Result (4 row(s)),"{""id"": ""S000:8000474C0>2>2>27H>S000:8000474C0"", ""amount"": 11.21, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46fR4>S000:8000474C0"", ""amount"": 65.15, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>46fR5>S000:8000474C0"", ""amount"": 579.25, ""currency"": ""US Dollar"", ""flag"": 0}{""id"": ""S000:8000474C0>2>2>4FSLO>S000:8000474C0"", ""amount"": 39883.41, ""currency"": ""US Dollar"", ""flag"": 0}"


Description,"3f bothE('transfer')bothE('transfer'): every transfer edge touching the account resolved by SEED (id=000:8000474C0) in either direction, combining 3d and 3e into one edge set."
Gremlin Query,"g.V(s).bothE('transfer').limit(10).project('id','amount').by(id).by('amount_paid')"
Result (10 row(s)),"{""id"": ""S000:8000474C0>2>2>27H>S000:8000474C0"", ""amount"": 11.21}{""id"": ""S000:8000474C0>2>2>46fR4>S000:8000474C0"", ""amount"": 65.15}{""id"": ""S000:8000474C0>2>2>46fR5>S000:8000474C0"", ""amount"": 579.25}{""id"": ""S000:8000474C0>2>2>46wl2>S0122705:80DB95010"", ""amount"": 289.99}{""id"": ""S000:8000474C0>2>2>46wl3>S0122705:80DB95010"", ""amount"": 912.44}{""id"": ""S000:8000474C0>2>2>48gLn>S00867:80124BD60"", ""amount"": 120864.29}{""id"": ""S000:8000474C0>2>2>48gLo>S00867:80124BD60"", ""amount"": 21145.52}{""id"": ""S000:8000474C0>2>2>48gLp>S00867:80124BD60"", ""amount"": 15718.12}{""id"": ""S000:8000474C0>2>2>48gLq>S00867:80124BD60"", ""amount"": 4216.02}{""id"": ""S000:8000474C0>2>2>4ATjX>S0154487:8212A2430"", ""amount"": 29855.52}"


Description,"3g outV() from edge rootoutV(): starting from a batch of transfer edges (not from one seed), jump to each edge's source vertex -- the payer for every sampled transfer."
Gremlin Query,"g.E().hasLabel('transfer').limit(10).outV().project('id','bank','acct').by(id).by('bank').by('acct')"
Result (10 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}"


Description,"3h inV() from edge rootinV(): same edge-root starting point as 3g, but jump to each edge's destination vertex -- the payee for every sampled transfer."
Gremlin Query,"g.E().hasLabel('transfer').limit(10).inV().project('id','bank','acct').by(id).by('bank').by('acct')"
Result (10 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""0154487:8212A2430"", ""bank"": ""0154487"", ""acct"": ""8212A2430""}"


Description,"3i bothV() from edge rootbothV(): from the same sampled edges, jump to both endpoints and deduplicate -- every account that appears anywhere in the sample, whether as payer or payee."
Gremlin Query,"g.E().hasLabel('transfer').limit(10).bothV().dedup().project('id','bank','acct').by(id).by('bank').by('acct')"
Result (4 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""0154487:8212A2430"", ""bank"": ""0154487"", ""acct"": ""8212A2430""}"


Description,"3j otherV()otherV(): starting from the own edges of the account resolved by SEED (id=000:8000474C0) (not an arbitrary edge batch), hop to whichever endpoint is NOT SEED -- a direction-agnostic way to list SEED's counterparties without needing to know in advance whether each edge is incoming or outgoing."
Gremlin Query,"g.V(s).bothE('transfer').limit(10).otherV().project('id','bank','acct').by(id).by('bank').by('acct')"
Result (10 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""0122705:80DB95010"", ""bank"": ""0122705"", ""acct"": ""80DB95010""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""00867:80124BD60"", ""bank"": ""00867"", ""acct"": ""80124BD60""}{""id"": ""0154487:8212A2430"", ""bank"": ""0154487"", ""acct"": ""8212A2430""}"


## 4 - Repeat / path steps


In [10]:
if SEED is not None:
    show(
        "g.V(s).out('transfer').simplePath().out('transfer').simplePath()"
        ".path().by(id).limit(10)",
        "4a 2-hop out simplePath",
        description=f"2-hop out-neighbors: accounts reached by two consecutive outgoing "
                     f"transfers from the account resolved by SEED (id={SEED}) (money-flow "
                     f"depth 2). Expressed as two chained out('transfer') calls instead of "
                     f"repeat().times(2) because HugeGraph's repeat() optimizer NPEs (see "
                     f"cell 15 below); simplePath() after each hop drops any path that "
                     f"revisits a vertex, so A->B->A is excluded. path().by(id) returns the "
                     f"full vertex-id chain for each hit.",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).both('transfer').simplePath().both('transfer').simplePath()"
        ".path().by(id).limit(10)",
        "4b 2-hop both simplePath",
        description=f"Same 2-hop idea as 4a (starting from the account resolved by SEED, "
                     f"id={SEED}) but using both('transfer') at each step, so it follows "
                     f"money in either direction -- paid-to or paid-by -- at each hop, "
                     f"surfacing indirect relationships regardless of transfer direction.",
        bindings={"s": SEED},
    )


Description,"4a 2-hop out simplePath2-hop out-neighbors: accounts reached by two consecutive outgoing transfers from the account resolved by SEED (id=000:8000474C0) (money-flow depth 2). Expressed as two chained out('transfer') calls instead of repeat().times(2) because HugeGraph's repeat() optimizer NPEs (see cell 15 below); simplePath() after each hop drops any path that revisits a vertex, so A->B->A is excluded. path().by(id) returns the full vertex-id chain for each hit."
Gremlin Query,g.V(s).out('transfer').simplePath().out('transfer').simplePath().path().by(id).limit(10)
Result (10 row(s)),"{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0270790:845399FE0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0270790:845399FE0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0153751:83FB88D10""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0153751:83FB88D10""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0153751:83FB88D10""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0281559:84ED5EA80""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0281559:84ED5EA80""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0025131:81F3DFBD0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0025131:81F3DFBD0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0025131:81F3DFBD0""]}"


Description,"4b 2-hop both simplePathSame 2-hop idea as 4a (starting from the account resolved by SEED, id=000:8000474C0) but using both('transfer') at each step, so it follows money in either direction -- paid-to or paid-by -- at each hop, surfacing indirect relationships regardless of transfer direction."
Gremlin Query,g.V(s).both('transfer').simplePath().both('transfer').simplePath().path().by(id).limit(10)
Result (10 row(s)),"{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0270790:845399FE0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0270790:845399FE0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0153751:83FB88D10""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0153751:83FB88D10""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0153751:83FB88D10""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0281559:84ED5EA80""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0281559:84ED5EA80""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0025131:81F3DFBD0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0025131:81F3DFBD0""]}{""labels"": [[], [], []], ""objects"": [""000:8000474C0"", ""0122705:80DB95010"", ""0025131:81F3DFBD0""]}"


## 5 - Aggregation (`count`, `sum`, `mean`, `groupCount`)


In [11]:
if SEED is not None:
    show("g.V(s).outE('transfer').count()", "5a out count",
         description=f"How many payments the account resolved by SEED (id={SEED}) made "
                      f"(outgoing transfer edge count).",
         bindings={"s": SEED})
    show("g.V(s).inE('transfer').count()", "5b in count",
         description=f"How many payments the account resolved by SEED (id={SEED}) "
                      f"received (incoming transfer edge count).",
         bindings={"s": SEED})
    show("g.V(s).outE('transfer').values('amount_paid').sum()", "5c sum(amount_paid)",
         description=f"Total amount the account resolved by SEED (id={SEED}) paid out "
                      f"across all its outgoing transfers.",
         bindings={"s": SEED})
    show("g.V(s).outE('transfer').values('amount_paid').mean()", "5d mean(amount_paid)",
         description=f"Average amount per outgoing transfer for the account resolved by "
                      f"SEED (id={SEED}).",
         bindings={"s": SEED})

show(
    f"g.V().hasLabel('account').limit({int(GROUPCOUNT_ACCOUNT_SCOPE_LIMIT)}).outE('transfer').limit({int(GROUPCOUNT_EDGE_SCOPE_LIMIT)})"
    ".groupCount().by('pay_currency').unfold()"
    ".project('currency','count')"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.keys))"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.values))"
    ".order().by(select('count'), desc).limit(10)",
    "5e groupCount by pay_currency (scoped)",
    description="groupCount().by('pay_currency'): tallies how many transfers use each "
                 "currency, over a scoped sample of accounts and their outgoing edges -- "
                 "a currency-mix breakdown, sorted from most to least common.",
)


Description,5a out countHow many payments the account resolved by SEED (id=000:8000474C0) made (outgoing transfer edge count).
Gremlin Query,g.V(s).outE('transfer').count()
Result (1 row(s)),23


Description,5b in countHow many payments the account resolved by SEED (id=000:8000474C0) received (incoming transfer edge count).
Gremlin Query,g.V(s).inE('transfer').count()
Result (1 row(s)),4


Description,5c sum(amount_paid)Total amount the account resolved by SEED (id=000:8000474C0) paid out across all its outgoing transfers.
Gremlin Query,g.V(s).outE('transfer').values('amount_paid').sum()
Result (1 row(s)),712672.4800000001


Description,5d mean(amount_paid)Average amount per outgoing transfer for the account resolved by SEED (id=000:8000474C0).
Gremlin Query,g.V(s).outE('transfer').values('amount_paid').mean()
Result (1 row(s)),30985.760000000006


Description,"5e groupCount by pay_currency (scoped)groupCount().by('pay_currency'): tallies how many transfers use each currency, over a scoped sample of accounts and their outgoing edges -- a currency-mix breakdown, sorted from most to least common."
Gremlin Query,"g.V().hasLabel('account').limit(300).outE('transfer').limit(5000).groupCount().by('pay_currency').unfold().project('currency','count').by(select(org.apache.tinkerpop.gremlin.structure.Column.keys)).by(select(org.apache.tinkerpop.gremlin.structure.Column.values)).order().by(select('count'), desc).limit(10)"
Result (6 row(s)),"{""currency"": ""US Dollar"", ""count"": 4253}{""currency"": ""Euro"", ""count"": 62}{""currency"": ""Yuan"", ""count"": 21}{""currency"": ""Yen"", ""count"": 10}{""currency"": ""Bitcoin"", ""count"": 5}{""currency"": ""Rupee"", ""count"": 1}"


## 6 - Projection (`values`, `project().by()`, `valueMap`, `identity`)


In [12]:
show("g.V().hasLabel('account').values('acct').limit(10)", "6a values('acct')",
     description="values('acct'): pull just the account-number property off each "
                  "'account' vertex, dropping everything else -- a flat list of values.")

show(
    "g.V().hasLabel('account').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')",
    "6b project().by() on vertices",
    description="project().by(): build a row per vertex with named fields (id, bank, "
                 "acct) instead of a raw property list -- the shape most later cells use.",
)

show("g.V().hasLabel('account').valueMap('bank','acct').limit(10)", "6c valueMap on vertices",
     description="valueMap('bank','acct'): return the requested properties as a map "
                  "keyed by property name (vertex id is not included by default) -- a "
                  "more compact alternative to project() when you don't need explicit "
                  "id/labels.")

show(
    "g.E().hasLabel('transfer').valueMap('amount_paid','pay_currency','is_laundering').limit(10)",
    "6d valueMap on edges",
    description="valueMap() on edges: same map-of-properties shape as 6c, but for "
                 "'transfer' edges -- amount, currency, and laundering flag per edge.",
)

show(
    "g.V().hasLabel('account').limit(20).project('account','out_deg','in_deg')"
    ".by(id).by(outE('transfer').count()).by(inE('transfer').count())"
    ".order().by(select('out_deg'), desc).limit(10)",
    "6e project with nested counts",
    description="project() with nested traversals: for each account, compute "
                 "out-degree and in-degree (how many transfers sent/received) inline via "
                 "nested outE()/inE().count() steps, then rank accounts by out-degree -- "
                 "surfaces the most active senders in a scoped sample.",
)


Description,"6a values('acct')values('acct'): pull just the account-number property off each 'account' vertex, dropping everything else -- a flat list of values."
Gremlin Query,g.V().hasLabel('account').values('acct').limit(10)
Result (10 row(s)),"""8000474C0""""800047930""""80006C140""""80006DFE0""""80006DFE1""""80006F150""""80006FE20""""8000719B0""""800071D00""""800071D50"""


Description,"6b project().by() on verticesproject().by(): build a row per vertex with named fields (id, bank, acct) instead of a raw property list -- the shape most later cells use."
Gremlin Query,"g.V().hasLabel('account').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')"
Result (10 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000"", ""acct"": ""8000474C0""}{""id"": ""000:800047930"", ""bank"": ""000"", ""acct"": ""800047930""}{""id"": ""000:80006C140"", ""bank"": ""000"", ""acct"": ""80006C140""}{""id"": ""000:80006DFE0"", ""bank"": ""000"", ""acct"": ""80006DFE0""}{""id"": ""000:80006DFE1"", ""bank"": ""000"", ""acct"": ""80006DFE1""}{""id"": ""000:80006F150"", ""bank"": ""000"", ""acct"": ""80006F150""}{""id"": ""000:80006FE20"", ""bank"": ""000"", ""acct"": ""80006FE20""}{""id"": ""000:8000719B0"", ""bank"": ""000"", ""acct"": ""8000719B0""}{""id"": ""000:800071D00"", ""bank"": ""000"", ""acct"": ""800071D00""}{""id"": ""000:800071D50"", ""bank"": ""000"", ""acct"": ""800071D50""}"


Description,"6c valueMap on verticesvalueMap('bank','acct'): return the requested properties as a map keyed by property name (vertex id is not included by default) -- a more compact alternative to project() when you don't need explicit id/labels."
Gremlin Query,"g.V().hasLabel('account').valueMap('bank','acct').limit(10)"
Result (10 row(s)),"{""bank"": [""000""], ""acct"": [""8000474C0""]}{""bank"": [""000""], ""acct"": [""800047930""]}{""bank"": [""000""], ""acct"": [""80006C140""]}{""bank"": [""000""], ""acct"": [""80006DFE0""]}{""bank"": [""000""], ""acct"": [""80006DFE1""]}{""bank"": [""000""], ""acct"": [""80006F150""]}{""bank"": [""000""], ""acct"": [""80006FE20""]}{""bank"": [""000""], ""acct"": [""8000719B0""]}{""bank"": [""000""], ""acct"": [""800071D00""]}{""bank"": [""000""], ""acct"": [""800071D50""]}"


Description,"6d valueMap on edgesvalueMap() on edges: same map-of-properties shape as 6c, but for 'transfer' edges -- amount, currency, and laundering flag per edge."
Gremlin Query,"g.E().hasLabel('transfer').valueMap('amount_paid','pay_currency','is_laundering').limit(10)"
Result (10 row(s)),"{""amount_paid"": 11.21, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 65.15, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 579.25, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 289.99, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 912.44, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 120864.29, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 21145.52, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 15718.12, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 4216.02, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}{""amount_paid"": 29855.52, ""pay_currency"": ""US Dollar"", ""is_laundering"": 0}"


Description,"6e project with nested countsproject() with nested traversals: for each account, compute out-degree and in-degree (how many transfers sent/received) inline via nested outE()/inE().count() steps, then rank accounts by out-degree -- surfaces the most active senders in a scoped sample."
Gremlin Query,"g.V().hasLabel('account').limit(20).project('account','out_deg','in_deg').by(id).by(outE('transfer').count()).by(inE('transfer').count()).order().by(select('out_deg'), desc).limit(10)"
Result (10 row(s)),"{""account"": ""000:800072CD0"", ""out_deg"": 59, ""in_deg"": 9}{""account"": ""000:800047930"", ""out_deg"": 56, ""in_deg"": 11}{""account"": ""000:800071D00"", ""out_deg"": 47, ""in_deg"": 6}{""account"": ""000:80006C140"", ""out_deg"": 46, ""in_deg"": 10}{""account"": ""000:800072650"", ""out_deg"": 37, ""in_deg"": 5}{""account"": ""000:800071D50"", ""out_deg"": 36, ""in_deg"": 5}{""account"": ""000:800073470"", ""out_deg"": 35, ""in_deg"": 4}{""account"": ""000:800074310"", ""out_deg"": 35, ""in_deg"": 9}{""account"": ""000:8000719B0"", ""out_deg"": 34, ""in_deg"": 5}{""account"": ""000:800073EF0"", ""out_deg"": 32, ""in_deg"": 1}"


## 7 - Ordering and paging (`order().by`, `limit`, `range`, `dedup`)


In [13]:
show(
    f"g.V().hasLabel('account').limit({int(RANGE_SCOPE_LIMIT)}).project('account','out_deg')"
    ".by(id).by(outE('transfer').count())"
    ".order().by(select('out_deg'), desc).limit(10)",
    "7a order by projected out degree",
    description="order().by(): rank accounts within a scoped sample by their "
                 "projected out-degree (number of outgoing transfers), highest first -- "
                 "'who sends the most payments' over that sample.",
)

show(
    f"g.V().hasLabel('account').limit({int(RANGE_SCOPE_LIMIT)}).order().by(id).range(0,10).id()",
    "7b page 1 via range(0,10) on scoped subset",
    description="range(0,10): the first page (rows 0-9) of a stable id-ordered scoped "
                 "subset -- demonstrates cursor-free pagination over a fixed ordering.",
)

show(
    f"g.V().hasLabel('account').limit({int(RANGE_SCOPE_LIMIT)}).order().by(id).range(10,20).id()",
    "7c page 2 via range(10,20) on scoped subset",
    description="range(10,20): the second page (rows 10-19) of that same ordering -- "
                 "paired with 7b to show consecutive pages don't overlap or skip.",
)

show(
    f"g.V().hasLabel('account').limit({int(DEDUP_SCOPE_LIMIT)}).both('transfer').dedup().count()",
    "7d dedup over neighbors (scoped)",
    description="dedup(): count of distinct neighbor accounts reachable from a scoped "
                 "subset via both('transfer') -- without dedup() the same neighbor "
                 "would be counted once per incoming account that connects to it.",
)


Description,"7a order by projected out degreeorder().by(): rank accounts within a scoped sample by their projected out-degree (number of outgoing transfers), highest first -- 'who sends the most payments' over that sample."
Gremlin Query,"g.V().hasLabel('account').limit(2000).project('account','out_deg').by(id).by(outE('transfer').count()).order().by(select('out_deg'), desc).limit(10)"
Result (10 row(s)),"{""account"": ""000:800072CD0"", ""out_deg"": 59}{""account"": ""000:800075150"", ""out_deg"": 58}{""account"": ""000:800047930"", ""out_deg"": 56}{""account"": ""000:8005EBCC0"", ""out_deg"": 49}{""account"": ""000:8001A3DD0"", ""out_deg"": 48}{""account"": ""000:800071D00"", ""out_deg"": 47}{""account"": ""000:80006C140"", ""out_deg"": 46}{""account"": ""000:8001D2C50"", ""out_deg"": 46}{""account"": ""000:8003D4990"", ""out_deg"": 45}{""account"": ""000:80024A670"", ""out_deg"": 42}"


Description,"7b page 1 via range(0,10) on scoped subsetrange(0,10): the first page (rows 0-9) of a stable id-ordered scoped subset -- demonstrates cursor-free pagination over a fixed ordering."
Gremlin Query,"g.V().hasLabel('account').limit(2000).order().by(id).range(0,10).id()"
Result (10 row(s)),"""000:8000474C0""""000:800047930""""000:80006C140""""000:80006DFE0""""000:80006DFE1""""000:80006F150""""000:80006FE20""""000:8000719B0""""000:800071D00""""000:800071D50"""


Description,"7c page 2 via range(10,20) on scoped subsetrange(10,20): the second page (rows 10-19) of that same ordering -- paired with 7b to show consecutive pages don't overlap or skip."
Gremlin Query,"g.V().hasLabel('account').limit(2000).order().by(id).range(10,20).id()"
Result (10 row(s)),"""000:800071D51""""000:800072650""""000:800072980""""000:800072CD0""""000:800073420""""000:800073470""""000:800073EF0""""000:800074310""""000:800074710""""000:800074AD0"""


Description,7d dedup over neighbors (scoped)dedup(): count of distinct neighbor accounts reachable from a scoped subset via both('transfer') -- without dedup() the same neighbor would be counted once per incoming account that connects to it.
Gremlin Query,g.V().hasLabel('account').limit(200).both('transfer').dedup().count()
Result (1 row(s)),1227


## 8 - Alias / select / where


In [14]:
show(
    f"g.V().hasLabel('account').limit({int(ALIAS_PAIR_SCOPE_LIMIT)}).as('a').out('transfer').as('b')"
    ".where('a', neq('b')).select('a','b').by(id).by(id).limit(10)",
    "8a as + select + where(neq)",
    description="as('a')...as('b') + where('a', neq('b')): label the start and "
                 "destination of each one-hop transfer, then keep only pairs where "
                 "they're actually different accounts (guards against self-transfer "
                 "edges), and select() both labeled ids back out as a pair.",
)

show(
    f"g.V().hasLabel('account').limit({int(FLAGGED_PROJECT_SCOPE_LIMIT)}).project('account','flagged_out')"
    ".by(id).by(outE('transfer').has('is_laundering',1).count())"
    ".where(select('flagged_out').is(gt(0)))"
    ".order().by(select('flagged_out'), desc).limit(10)",
    "8b where(select(...).is(gt(...)))",
    description="where(select('flagged_out').is(gt(0))): compute a "
                 "flagged-outgoing-count per account via project(), then filter down to "
                 "only accounts where that count is positive -- i.e. accounts that have "
                 "sent at least one laundering-flagged transfer, ranked by how many "
                 "they've sent.",
)


Description,"8a as + select + where(neq)as('a')...as('b') + where('a', neq('b')): label the start and destination of each one-hop transfer, then keep only pairs where they're actually different accounts (guards against self-transfer edges), and select() both labeled ids back out as a pair."
Gremlin Query,"g.V().hasLabel('account').limit(200).as('a').out('transfer').as('b').where('a', neq('b')).select('a','b').by(id).by(id).limit(10)"
Result (10 row(s)),"{""a"": ""000:8000474C0"", ""b"": ""0122705:80DB95010""}{""a"": ""000:8000474C0"", ""b"": ""0122705:80DB95010""}{""a"": ""000:8000474C0"", ""b"": ""00867:80124BD60""}{""a"": ""000:8000474C0"", ""b"": ""00867:80124BD60""}{""a"": ""000:8000474C0"", ""b"": ""00867:80124BD60""}{""a"": ""000:8000474C0"", ""b"": ""00867:80124BD60""}{""a"": ""000:8000474C0"", ""b"": ""0154487:8212A2430""}{""a"": ""000:8000474C0"", ""b"": ""0154487:8212A2430""}{""a"": ""000:8000474C0"", ""b"": ""01208:80013C440""}{""a"": ""000:8000474C0"", ""b"": ""01208:80013C440""}"


Description,"8b where(select(...).is(gt(...)))where(select('flagged_out').is(gt(0))): compute a flagged-outgoing-count per account via project(), then filter down to only accounts where that count is positive -- i.e. accounts that have sent at least one laundering-flagged transfer, ranked by how many they've sent."
Gremlin Query,"g.V().hasLabel('account').limit(1000).project('account','flagged_out').by(id).by(outE('transfer').has('is_laundering',1).count()).where(select('flagged_out').is(gt(0))).order().by(select('flagged_out'), desc).limit(10)"
Result (1 row(s)),"{""account"": ""000:800D81CB0"", ""flagged_out"": 1}"


## 9 - Compound `where` (`and`, `or`, `not`)


In [15]:
show(
    f"g.V().hasLabel('account').limit({int(COMPOUND_WHERE_SCOPE_LIMIT)})"
    ".where(__.and(__.outE('transfer').count().is(gt(0)), __.inE('transfer').count().is(gt(0))))"
    ".project('account','in_deg','out_deg')"
    ".by(id).by(inE('transfer').count()).by(outE('transfer').count())"
    ".limit(10)",
    "9a where(__.and(...))",
    description="where(__.and(...)): keep only accounts that have BOTH at least one "
                 "outgoing AND at least one incoming transfer -- i.e. accounts acting as "
                 "a pass-through/intermediary rather than a pure source or pure sink.",
)

show(
    f"g.V().hasLabel('account').limit({int(COMPOUND_WHERE_SCOPE_LIMIT)})"
    ".where(__.or(__.outE('transfer').has('is_laundering',1), __.inE('transfer').has('is_laundering',1)))"
    ".project('account','flagged_in','flagged_out')"
    ".by(id)"
    ".by(inE('transfer').has('is_laundering',1).count())"
    ".by(outE('transfer').has('is_laundering',1).count())"
    ".limit(10)",
    "9b where(__.or(...))",
    description="where(__.or(...)): keep accounts with a flagged transfer in EITHER "
                 "direction -- broader than 2b/8b, since it catches accounts flagged as "
                 "the receiver of laundering money too, not just the sender.",
)

show(
    f"g.V().hasLabel('account').limit({int(COMPOUND_WHERE_SCOPE_LIMIT)}).where(__.not(__.outE('transfer'))).id().limit(10)",
    "9c where(__.not(...))",
    description="where(__.not(...)): the inverse of 9a's outgoing check -- accounts "
                 "with zero outgoing transfers at all, i.e. pure receivers/dead-end "
                 "accounts.",
)


Description,9a where(__.and(...))where(__.and(...)): keep only accounts that have BOTH at least one outgoing AND at least one incoming transfer -- i.e. accounts acting as a pass-through/intermediary rather than a pure source or pure sink.
Gremlin Query,"g.V().hasLabel('account').limit(1000).where(__.and(__.outE('transfer').count().is(gt(0)), __.inE('transfer').count().is(gt(0)))).project('account','in_deg','out_deg').by(id).by(inE('transfer').count()).by(outE('transfer').count()).limit(10)"
Result (10 row(s)),"{""account"": ""000:8000474C0"", ""in_deg"": 4, ""out_deg"": 23}{""account"": ""000:800047930"", ""in_deg"": 11, ""out_deg"": 56}{""account"": ""000:80006C140"", ""in_deg"": 10, ""out_deg"": 46}{""account"": ""000:80006DFE0"", ""in_deg"": 5, ""out_deg"": 30}{""account"": ""000:80006F150"", ""in_deg"": 4, ""out_deg"": 24}{""account"": ""000:80006FE20"", ""in_deg"": 6, ""out_deg"": 29}{""account"": ""000:8000719B0"", ""in_deg"": 5, ""out_deg"": 34}{""account"": ""000:800071D00"", ""in_deg"": 6, ""out_deg"": 47}{""account"": ""000:800071D50"", ""in_deg"": 5, ""out_deg"": 36}{""account"": ""000:800072650"", ""in_deg"": 5, ""out_deg"": 37}"


Description,"9b where(__.or(...))where(__.or(...)): keep accounts with a flagged transfer in EITHER direction -- broader than 2b/8b, since it catches accounts flagged as the receiver of laundering money too, not just the sender."
Gremlin Query,"g.V().hasLabel('account').limit(1000).where(__.or(__.outE('transfer').has('is_laundering',1), __.inE('transfer').has('is_laundering',1))).project('account','flagged_in','flagged_out').by(id).by(inE('transfer').has('is_laundering',1).count()).by(outE('transfer').has('is_laundering',1).count()).limit(10)"
Result (3 row(s)),"{""account"": ""000:800648330"", ""flagged_in"": 1, ""flagged_out"": 0}{""account"": ""000:800802660"", ""flagged_in"": 1, ""flagged_out"": 0}{""account"": ""000:800D81CB0"", ""flagged_in"": 0, ""flagged_out"": 1}"


Description,"9c where(__.not(...))where(__.not(...)): the inverse of 9a's outgoing check -- accounts with zero outgoing transfers at all, i.e. pure receivers/dead-end accounts."
Gremlin Query,g.V().hasLabel('account').limit(1000).where(__.not(__.outE('transfer'))).id().limit(10)
Result (7 row(s)),"""000:800191D80""""000:8002448A0""""000:800249FD0""""000:800335BA0""""000:8003663A0""""000:8003F3CC0""""000:8011F7A20"""


## 10 - `groupCount` with alias key


In [16]:
show(
    f"g.V().hasLabel('account').limit({int(BANK_GROUP_SCOPE_LIMIT)}).as('a').groupCount()"
    ".by(select('a').by('bank')).unfold()"
    ".project('bank','accounts')"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.keys))"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.values))"
    ".order().by(select('accounts'), desc).limit(10)",
    "10a bank-level account counts",
    description="groupCount().by(select('a').by('bank')): label each scoped account "
                 "'a', then group-count by that account's bank property -- i.e. how "
                 "many accounts belong to each bank, sorted from largest to smallest.",
)


Description,"10a bank-level account countsgroupCount().by(select('a').by('bank')): label each scoped account 'a', then group-count by that account's bank property -- i.e. how many accounts belong to each bank, sorted from largest to smallest."
Gremlin Query,"g.V().hasLabel('account').limit(5000).as('a').groupCount().by(select('a').by('bank')).unfold().project('bank','accounts').by(select(org.apache.tinkerpop.gremlin.structure.Column.keys)).by(select(org.apache.tinkerpop.gremlin.structure.Column.values)).order().by(select('accounts'), desc).limit(10)"
Result (3 row(s)),"{""bank"": ""000"", ""accounts"": 2722}{""bank"": ""001"", ""accounts"": 1280}{""bank"": ""002"", ""accounts"": 998}"


## 11 - Edge-root projection with `outV` / `inV`


In [17]:
show(
    "g.E().hasLabel('transfer').limit(15)"
    ".project('from','to','amount','currency','format','flag','ts')"
    ".by(outV().id()).by(inV().id())"
    ".by('amount_paid').by('pay_currency').by('pay_format')"
    ".by('is_laundering').by('ts')",
    "11a edge-root projection",
    description="project() starting directly from the edge root (g.E(), not g.V()): "
                 "flattens each 'transfer' edge into one full record -- sender, "
                 "receiver, amount, currency, payment format, laundering flag, and "
                 "timestamp -- a transaction-log-style view rather than an "
                 "account-centric one.",
)


## 12 - `simplePath` across multi-hop traversals


In [18]:
if SEED is not None:
    show(
        "g.V(s).out('transfer').simplePath().out('transfer').simplePath()"
        ".out('transfer').simplePath()"
        ".path().by(id).limit(10)",
        "12a 3-hop simplePath",
        description=f"3-hop out-neighbors: accounts reached by three consecutive outgoing "
                     f"transfers from the account resolved by SEED (id={SEED}) (money-flow "
                     f"depth 3), same explicit-chaining pattern as cell 4 (see cell 4/15 "
                     f"note on why repeat() is avoided). simplePath() after every hop "
                     f"prevents the walk from doubling back through a vertex it already "
                     f"visited.",
        bindings={"s": SEED},
    )


Description,"12a 3-hop simplePath3-hop out-neighbors: accounts reached by three consecutive outgoing transfers from the account resolved by SEED (id=000:8000474C0) (money-flow depth 3), same explicit-chaining pattern as cell 4 (see cell 4/15 note on why repeat() is avoided). simplePath() after every hop prevents the walk from doubling back through a vertex it already visited."
Gremlin Query,g.V(s).out('transfer').simplePath().out('transfer').simplePath().out('transfer').simplePath().path().by(id).limit(10)
Result (10 row(s)),"{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}{""labels"": [[], [], [], []], ""objects"": [""000:8000474C0"", ""00867:80124BD60"", ""0215309:809025AB0"", ""0189342:83DFABCB0""]}"


## 13 - `identity()` as no-op and in projection


In [19]:
show(
    "g.V().hasLabel('account').limit(5).identity().project('id','bank').by(id).by('bank')",
    "13a identity() no-op",
    description="identity(): a step that passes each element through unchanged -- "
                 "demonstrates that inserting it into a traversal chain has no effect "
                 "on the result, useful as a placeholder/no-op when building traversals "
                 "programmatically.",
)

show(
    "g.V().hasLabel('account').limit(5).project('id','self').by(id).by(identity())",
    "13b by(identity())",
    description="by(identity()): inside project(), identity() returns the element "
                 "itself (here reduced to its id) as the field value -- a way to "
                 "include 'the current element' as one of several projected columns.",
)


Description,"13a identity() no-opidentity(): a step that passes each element through unchanged -- demonstrates that inserting it into a traversal chain has no effect on the result, useful as a placeholder/no-op when building traversals programmatically."
Gremlin Query,"g.V().hasLabel('account').limit(5).identity().project('id','bank').by(id).by('bank')"
Result (5 row(s)),"{""id"": ""000:8000474C0"", ""bank"": ""000""}{""id"": ""000:800047930"", ""bank"": ""000""}{""id"": ""000:80006C140"", ""bank"": ""000""}{""id"": ""000:80006DFE0"", ""bank"": ""000""}{""id"": ""000:80006DFE1"", ""bank"": ""000""}"


Description,"13b by(identity())by(identity()): inside project(), identity() returns the element itself (here reduced to its id) as the field value -- a way to include 'the current element' as one of several projected columns."
Gremlin Query,"g.V().hasLabel('account').limit(5).project('id','self').by(id).by(identity())"
Result (5 row(s)),"{""id"": ""000:8000474C0"", ""self"": {""id"": ""000:8000474C0"", ""label"": ""account"", ""type"": ""vertex"", ""properties"": {""bank"": ""000"", ""acct"": ""8000474C0""}}}{""id"": ""000:800047930"", ""self"": {""id"": ""000:800047930"", ""label"": ""account"", ""type"": ""vertex"", ""properties"": {""bank"": ""000"", ""acct"": ""800047930""}}}{""id"": ""000:80006C140"", ""self"": {""id"": ""000:80006C140"", ""label"": ""account"", ""type"": ""vertex"", ""properties"": {""bank"": ""000"", ""acct"": ""80006C140""}}}{""id"": ""000:80006DFE0"", ""self"": {""id"": ""000:80006DFE0"", ""label"": ""account"", ""type"": ""vertex"", ""properties"": {""bank"": ""000"", ""acct"": ""80006DFE0""}}}{""id"": ""000:80006DFE1"", ""self"": {""id"": ""000:80006DFE1"", ""label"": ""account"", ""type"": ""vertex"", ""properties"": {""bank"": ""000"", ""acct"": ""80006DFE1""}}}"


## 14 - Bounded account-activity ranking query

This is a bounded account-activity ranking query based on the `is_laundering` edge flag.


In [20]:
show(
    f"g.V().hasLabel('account').limit({int(RANKING_SCOPE_LIMIT)})"
    ".where(outE('transfer').has('is_laundering',1))"
    ".project('account','bank','flagged_out','total_out','fan_out')"
    ".by(id).by('bank')"
    ".by(outE('transfer').has('is_laundering',1).count())"
    ".by(outE('transfer').count())"
    ".by(out('transfer').dedup().count())"
    ".order().by(select('flagged_out'), desc).limit(20)",
    "14a account ranking by flagged outgoing transfers",
    description="where(outE(...).has('is_laundering',1)): keep only accounts with at "
                 "least one flagged outgoing transfer, then project a full risk profile "
                 "per account -- flagged-out count, total outgoing transfers, and "
                 "distinct fan-out (how many different accounts it paid) -- ranked by "
                 "flagged-out count so the highest-risk senders surface first.",
)


Description,"14a account ranking by flagged outgoing transferswhere(outE(...).has('is_laundering',1)): keep only accounts with at least one flagged outgoing transfer, then project a full risk profile per account -- flagged-out count, total outgoing transfers, and distinct fan-out (how many different accounts it paid) -- ranked by flagged-out count so the highest-risk senders surface first."
Gremlin Query,"g.V().hasLabel('account').limit(3000).where(outE('transfer').has('is_laundering',1)).project('account','bank','flagged_out','total_out','fan_out').by(id).by('bank').by(outE('transfer').has('is_laundering',1).count()).by(outE('transfer').count()).by(out('transfer').dedup().count()).order().by(select('flagged_out'), desc).limit(20)"
Result (2 row(s)),"{""account"": ""000:800D81CB0"", ""bank"": ""000"", ""flagged_out"": 1, ""total_out"": 11, ""fan_out"": 5}{""account"": ""000:8028D5120"", ""bank"": ""000"", ""flagged_out"": 1, ""total_out"": 4, ""fan_out"": 3}"


## 15 - Deep multi-hop traversals (bounded for HugeGraph)

These cells are tuned for runtime safety:
- use a single seed
- keep strict limits
- avoid unbounded global edge scans


In [21]:
# HUB: pick the flagged-laundering seed if one exists, otherwise fall
# back to the general SEED. Deep multi-hop traversals below need a
# vertex with enough real fan-out to produce interesting results --
# a randomly chosen SEED could dead-end after one or two hops.
HUB = FLAGGED_SEED or SEED
print("Deep-traversal HUB:", HUB)


def chained_hops(step, hops, per_hop_limit=None, simple_path=False):
    # Builds a Gremlin fragment that repeats `step` (e.g. "out('transfer')")
    # `hops` times back-to-back, joined by '.', as an explicit substitute
    # for repeat(step).times(hops). HugeGraph's repeat()/times() hits a
    # NullPointerException in the RepeatUnrollStrategy optimizer
    # (Query.hashCode() dereferences an uninitialized resultType before
    # the query is ever built) -- chaining the step out by hand produces
    # a traversal with no repeat() step at all, so that optimizer path is
    # never triggered. per_hop_limit caps fan-out at every single hop
    # (not just the final result) to keep exponential blow-up bounded;
    # simple_path excludes any hop that revisits an already-seen vertex.
    parts = []
    for _ in range(hops):
        hop = step
        if per_hop_limit is not None:
            hop += f".limit({int(per_hop_limit)})"
        if simple_path:
            hop += ".simplePath()"
        parts.append(hop)
    return ".".join(parts)


if HUB is not None:
    show(
        "g.V(h)."
        + chained_hops("out('transfer')", 5, simple_path=True)
        + ".path().by(id).limit(10)",
        "15a 5-hop path sample",
        description=f"Sample of concrete 5-hop money-flow paths starting at the account "
                     f"resolved by HUB (id={HUB}) -- five consecutive outgoing transfers, "
                     f"with simplePath() at each hop so no path loops back through a "
                     f"vertex it already passed through. Returns the full vertex-id chain "
                     f"(path().by(id)) for up to 10 such paths, i.e. concrete examples of "
                     f"'money moved through 5 accounts starting at HUB'.",
        bindings={"h": HUB},
    )

if HUB is not None:
    show(
        "g.V(h)."
        + chained_hops("out('transfer')", 8, per_hop_limit=HOP8_OUT_LIMIT)
        + f".dedup().limit({int(HOP8_RESULT_LIMIT)}).count()",
        "15b distinct accounts reachable in 8 hops (scoped)",
        description=f"How many distinct accounts are reachable from the account resolved "
                     f"by HUB (id={HUB}) within 8 hops of outgoing transfers -- a "
                     f"reach/exposure metric ('how far could HUB's money have spread'). "
                     f"Each individual hop's fan-out is capped at HOP8_OUT_LIMIT to keep "
                     f"the combinatorial explosion bounded (8 hops of unbounded fan-out "
                     f"would be intractable), then dedup() collapses accounts reached via "
                     f"multiple routes before counting (also capped at HOP8_RESULT_LIMIT "
                     f"so the dedup+count itself stays bounded).",
        bindings={"h": HUB},
    )

seed_rows = show(
    "g.E().hasLabel('transfer').has('is_laundering',1)"
    ".groupCount().by(outV().id()).unfold()"
    ".project('seed','flagged_out_edges')"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.keys))"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.values))"
    ".order().by(select('flagged_out_edges'), desc).limit(5)",
    "15c accounts with highest flagged outgoing transfers",
    description="Rank accounts by how many laundering-flagged transfers they sent, "
                 "take the top 5 -- these become the seed set for the per-seed 6-hop "
                 "reachability check in 15d below (i.e. 'start the deep search from the "
                 "most suspicious senders, not an arbitrary vertex').",
)

if seed_rows:
    print("\n15d 6-hop reachability per seed")
    print(
        "For each top-flagged seed: distinct accounts reachable within 6 outgoing "
        "hops (same bounded-fan-out + dedup pattern as 15b, via "
        "HOP6_OUT_LIMIT/HOP6_RESULT_LIMIT) -- comparing each seed's flagged-edge "
        "count against its 6-hop reach gives a rough sense of which suspicious "
        "accounts also have the widest downstream spread. Retries once on a "
        "transient Gremlin alias-binding error before giving up on that seed."
    )
    seed_scoped_query = (
        "g.V(s)."
        + chained_hops("out('transfer')", 6, per_hop_limit=HOP6_OUT_LIMIT)
        + f".dedup().limit({int(HOP6_RESULT_LIMIT)}).count()"
    )
    for row in seed_rows:
        s = row.get("seed")
        try:
            c = gremlin(seed_scoped_query, bindings={"s": s}, eval_timeout_ms=120_000)
            cnt = c[0] if c else 0
            print(
                f"  seed={s}  flagged_out_edges={row.get('flagged_out_edges')} "
                f"6hop_scoped={cnt}"
            )
        except Exception as e:
            err_text = str(e)
            if needs_alias_fallback(err_text):
                try:
                    wait_for_gremlin_alias_binding(
                        timeout_sec=40,
                        poll_sec=2,
                        required_consecutive=2,
                    )
                    c = gremlin(seed_scoped_query, bindings={"s": s}, eval_timeout_ms=120_000)
                    cnt = c[0] if c else 0
                    print(
                        f"  seed={s}  flagged_out_edges={row.get('flagged_out_edges')} "
                        f"6hop_scoped={cnt} (after alias rebind)"
                    )
                    continue
                except Exception as e2:
                    print(f"  seed={s} FAILED (scoped after alias rebind): {e2}")
                    continue
            print(f"  seed={s} FAILED (scoped): {e}")


Deep-traversal HUB: 000:800D81CB0


Description,"15a 5-hop path sampleSample of concrete 5-hop money-flow paths starting at the account resolved by HUB (id=000:800D81CB0) -- five consecutive outgoing transfers, with simplePath() at each hop so no path loops back through a vertex it already passed through. Returns the full vertex-id chain (path().by(id)) for up to 10 such paths, i.e. concrete examples of 'money moved through 5 accounts starting at HUB'."
Gremlin Query,g.V(h).out('transfer').simplePath().out('transfer').simplePath().out('transfer').simplePath().out('transfer').simplePath().out('transfer').simplePath().path().by(id).limit(10)
Result (10 row(s)),"{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}{""labels"": [[], [], [], [], [], []], ""objects"": [""000:800D81CB0"", ""021418:8014E3810"", ""004495:8028467C0"", ""0113044:820501110"", ""02106128:83289D3C0"", ""01111161:83A4F9300""]}"


Description,"15b distinct accounts reachable in 8 hops (scoped)How many distinct accounts are reachable from the account resolved by HUB (id=000:800D81CB0) within 8 hops of outgoing transfers -- a reach/exposure metric ('how far could HUB's money have spread'). Each individual hop's fan-out is capped at HOP8_OUT_LIMIT to keep the combinatorial explosion bounded (8 hops of unbounded fan-out would be intractable), then dedup() collapses accounts reached via multiple routes before counting (also capped at HOP8_RESULT_LIMIT so the dedup+count itself stays bounded)."
Gremlin Query,g.V(h).out('transfer').limit(1000).out('transfer').limit(1000).out('transfer').limit(1000).out('transfer').limit(1000).out('transfer').limit(1000).out('transfer').limit(1000).out('transfer').limit(1000).out('transfer').limit(1000).dedup().limit(50000).count()
Result (1 row(s)),37


Description,"15c accounts with highest flagged outgoing transfersRank accounts by how many laundering-flagged transfers they sent, take the top 5 -- these become the seed set for the per-seed 6-hop reachability check in 15d below (i.e. 'start the deep search from the most suspicious senders, not an arbitrary vertex')."
Gremlin Query,"g.E().hasLabel('transfer').has('is_laundering',1).groupCount().by(outV().id()).unfold().project('seed','flagged_out_edges').by(select(org.apache.tinkerpop.gremlin.structure.Column.keys)).by(select(org.apache.tinkerpop.gremlin.structure.Column.values)).order().by(select('flagged_out_edges'), desc).limit(5)"
Result (5 row(s)),"{""seed"": ""070:10042B660"", ""flagged_out_edges"": 169}{""seed"": ""070:10042B6A8"", ""flagged_out_edges"": 107}{""seed"": ""070:10042B6F0"", ""flagged_out_edges"": 36}{""seed"": ""070:10042B780"", ""flagged_out_edges"": 25}{""seed"": ""070:10042B7C8"", ""flagged_out_edges"": 18}"
